# Mission 3 ver5 — 기존 모델 재사용 + 증상별 단어 attention

**GPU 런타임을 선택하고 위에서부터 모두 실행하세요.** Drive 연결 창이 나오면 연결합니다.
실행 코드가 포함되어 있으므로 별도 `.py` 업로드는 필요 없습니다.
**기존 ver4 결과 폴더 전체와 서울 Training JSON은 Drive에 있어야 합니다.**

ver4의 최적 모델 5개와 보정용 예측을 재사용하고, 구조를 개선한 모델 **1개만 추가 학습**합니다.
새 모델은 fold 1 최적 가중치에서 시작하며 **encoder 고정 1 epoch + 전체 미세조정 최대 3 epoch**를 수행합니다.
다섯 모델을 처음부터 다시 학습하거나 여러 학습률을 장시간 탐색하지 않습니다.

이전 A100 학습 기록은 22 epoch, 학습+stop 평가 합 630.28분입니다.
ver5 전체 실행은 **약 2~4시간을 계획용으로 예상**하지만 아직 실측이 아니며,
GPU·데이터 읽기·약 4GB 재개 파일 저장·기본 모델 복사·최종 추론에 따라 달라집니다.
로그의 `train_ETA_minutes`는 현재 epoch의 학습 잔여 시간이며 평가·저장 시간은 제외합니다.
기존 ver4 외에 **약 12GiB 이상의 추가 저장 공간**을 준비하세요.

성능은 실행 후 확인해야 합니다. Training 내부 교차평가에서 새 구성이 더 좋지 않으면
**기존 ver4 보정 구성을 선택**합니다. 이 선택이 실제 Test 성능의 향상을 보장하지는 않습니다.


## 논문 전체 검토와 적용

`mission3_ver5`의 PDF 3개는 **서로 다른 논문 2편**입니다.
`1-s2.0-S2405844022009719-main.pdf`와
`Automatic-symptoms-identification-from-a-massive-volume-of-unstruct_2022_Hel.pdf`는
12페이지 텍스트와 삽입 이미지가 같은 사본입니다. `2403.06734v1.pdf`는 별도의 13페이지 논문입니다.

| 연구 | 논문에서 확인한 내용 | ver5에 적용 |
|---|---|---|
| [Faris et al., Heliyon 2022](https://doi.org/10.1016/j.heliyon.2022.e09683), pp.4,6–8 | 동결한 AraBERT 특징과 작은 분류기로 아랍어 증상 추천 | 한국어 ver4 encoder를 재사용하고 새 head를 먼저 동결 encoder 위에서 학습 |
| [CognitiveEMS, 2024 v1](https://arxiv.org/abs/2403.06734), pp.6–7,10 | label마다 서로 다른 token 증거를 보는 attention과 지식 융합 | 9개 증상별 token attention을 기존 global/window head에 추가 |

이는 논문을 Mission 3에 맞춘 **설계 응용**이며 원 논문의 완전한 재현은 아닙니다.
기존 ver4는 CLS/평균으로 window를 압축한 뒤 window 사이에서만 증상별 attention을 했습니다.
ver5는 짧은 대화에서도 증상마다 중요한 token을 직접 선택하고, 전체 문맥도 함께 사용합니다.
추가 출력은 0으로 초기화하여 기존 모델에서 출발합니다.

영어 TinyClinicalBERT·외부 의료 지식 그래프·합성 데이터/API, 상위 5개 강제 출력,
희소 증상 제거, 부정 표현/숫자 삭제, 증상 자동 재라벨링은 적용하지 않습니다.
두 논문의 대상 언어·라벨 수·평가 지표가 달라 그 점수를 우리 Macro F1과 직접 비교할 수 없습니다.
전체 검토 근거는 실행 폴더와 제출 코드 폴더에 포함된 `RESEARCH.md`에 남깁니다.


In [ ]:
# 1. 기존 폴더 구조라면 수정하지 않고 모두 실행하세요.
BASE_DIR = None          # ver4 전체 결과 폴더. 예: /content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4
TRAIN_LABEL_DIR = None   # TL_서울_구급 폴더 자동 탐색
VALID_LABEL_DIR = None   # VL_서울_구급 자동 탐색. 다른 Test JSON 폴더를 직접 지정해도 됩니다.
OUTPUT_DIR = None        # 기본값: 기존 ver4 폴더와 나란한 results/mission3/ver5
RUN_VALIDATION = True    # 학습/보정 종료 후 예측. 정답이 있으면 평가도 수행합니다.

TRAIN_CONFIG = {
    "warmup_epochs": 1,
    "finetune_epochs": 3,
    "patience": 2,
}
# 기본은 GPU 메모리에 따라 batch/window를 8/8, 4/4, 2/2로 선택하고 유효 batch 32를 유지합니다.
# 수동 예: TRAIN_CONFIG.update(batch_size=2, window_batch_size=2, gradient_accumulation_steps=16)
# 학습 설정을 바꾸는 새 실험은 OUTPUT_DIR를 새 폴더로 지정하세요.
INFERENCE_BATCH_SIZE = 8


In [ ]:
# 2. 저장 모델과 호환되는 패키지. 이미 호환되는 Colab torch는 유지합니다.
import subprocess
import sys

REQUIREMENTS = ['torch>=2.6,<3', 'transformers==4.57.1', 'numpy==2.1.3', 'scikit-learn==1.6.1', 'scipy>=1.11,<2', 'iterative-stratification==0.1.9', 'joblib>=1.3,<2']
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *REQUIREMENTS], check=True)
print("환경 준비 완료. 학습/추론은 새 Python 프로세스에서 실행하므로 패키지 설치 후 커널 재시작이 필요하지 않습니다.")


In [ ]:
# 3. Drive 연결과 경로 확인
import json
import os
import tempfile
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

START_DIR = Path.cwd().resolve()
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
roots = ([Path("/content"), Path("/content/drive/MyDrive")] if IN_COLAB else [])
roots += [START_DIR, *list(START_DIR.parents)[:6]]
roots = list(dict.fromkeys(roots))

def locate(explicit, candidates, valid, name, required=True):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        if not valid(path):
            raise FileNotFoundError(f"{name} 경로를 확인하세요: {path}")
        return path
    for path in dict.fromkeys(candidates):
        if valid(path):
            return path.resolve()
    if required:
        raise FileNotFoundError(f"{name}을 찾지 못했습니다. 첫 설정 셀에 실제 폴더 경로를 지정하세요.")
    return None

base_candidates = []
for root in roots:
    base_candidates.extend([root, root / "results/mission3/ver4", root / "DCC_AIdeas/results/mission3/ver4"])
BASE_DIR = locate(BASE_DIR, base_candidates, lambda p: (p / "mission3_ver4_model.ckpt").is_file(), "BASE_DIR")
for name in ("splits.json", "calibration_predictions.npz", "sparse.joblib", "tokenizer"):
    if not (BASE_DIR / name).exists():
        raise FileNotFoundError(f"기존 ver4 전체 결과 묶음이 필요합니다: {BASE_DIR / name}")

def label_candidates(name, split):
    values = []
    for root in [*roots, *list(BASE_DIR.parents)[:4]]:
        short = "train" if split == "Training" else "validation"
        values.extend([root / name, root / "mission3_data" / short / name])
        for base in [root, root / "DATA", root / "data", root / "DCC_AIdeas/DATA", root / "DCC_AIdeas/data"]:
            for prefix in [base, base / "대학부 데이터"]:
                values.append(prefix / split / "2.라벨링데이터" / name)
    return values

has_json = lambda p: p.is_dir() and next(p.rglob("*.json"), None) is not None
TRAIN_LABEL_DIR = locate(TRAIN_LABEL_DIR, label_candidates("TL_서울_구급", "Training"), has_json, "TRAIN_LABEL_DIR")
VALID_LABEL_DIR = locate(VALID_LABEL_DIR, label_candidates("VL_서울_구급", "Validation"), has_json, "VALID_LABEL_DIR", required=False)
OUTPUT_DIR = (BASE_DIR.parent / "ver5" if OUTPUT_DIR is None else Path(OUTPUT_DIR).expanduser()).resolve()
if OUTPUT_DIR == BASE_DIR or OUTPUT_DIR.is_relative_to(BASE_DIR) or BASE_DIR.is_relative_to(OUTPUT_DIR):
    raise ValueError("OUTPUT_DIR는 기존 ver4와 별도 폴더여야 합니다.")
print("기존 모델:", BASE_DIR)
print("Training:", TRAIN_LABEL_DIR)
print("최종 예측 입력:", VALID_LABEL_DIR)
print("ver5 결과:", OUTPUT_DIR)


In [ ]:
# @title 4. 포함된 ver5 실행 코드 준비 (수정 없이 실행)
RUNTIME_SOURCES = {'__init__.py': '"""Mission 3 ver5: warm-start token evidence model and calibrated ensemble."""\n', 'neural.py': '"""Warm-started token-evidence residual and completed-epoch training resume.\n\nThe saved ver4 encoder/global head is retained. A zero-initialized residual\nadds label-specific token evidence without changing initial inference logits.\nResume restores the last *completed* epoch; an interrupted epoch is replayed.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport tempfile\nimport time\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\nif __package__:\n    from .ver4_runtime.neural import (\n        DialogueClassifier as LegacyDialogueClassifier,\n        DialogueCollator, DialogueDataset, NUM_LABELS, _amp, _device, _loader,\n        _optimizer, _predict, _targets, tokenize_dialogues,\n    )\nelse:\n    from ver4_runtime.neural import (\n        DialogueClassifier as LegacyDialogueClassifier,\n        DialogueCollator, DialogueDataset, NUM_LABELS, _amp, _device, _loader,\n        _optimizer, _predict, _targets, tokenize_dialogues,\n    )\n\n\nMODEL_FORMAT = "mission3_ver5_neural"\nLAST_FORMAT = "mission3_ver5_last"\nDEFAULTS = {\n    "warmup_epochs": 1, "finetune_epochs": 3, "patience": 2,\n    "learning_rate": 5e-6, "head_learning_rate": 1e-4,\n    "warmup_head_learning_rate": 3e-4, "attention_dim": 256, "dropout": 0.2,\n    "batch_size": 2, "window_batch_size": 2,\n    "gradient_accumulation_steps": 16, "inference_batch_size": 8,\n    "gradient_checkpointing": True, "device": "cuda", "log_every": 100,\n    "layerwise_lr_decay": 0.95, "weight_decay": 0.01,\n    "pos_weight_power": 0.5, "warmup_ratio": 0.1,\n}\n\n\nclass DialogueClassifier(LegacyDialogueClassifier):\n    """One encoder forward supplies both global and label-wise token evidence."""\n\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=8, attention_dim=256, special_token_ids=()):\n        super().__init__(encoder, num_labels, dropout, window_batch_size)\n        if int(attention_dim) < 1:\n            raise ValueError("attention_dim must be positive")\n        self.attention_dim = int(attention_dim)\n        self.special_token_ids = tuple(sorted(set(int(x) for x in special_token_ids)))\n        self.token_projection = nn.Sequential(\n            nn.Linear(encoder.config.hidden_size, self.attention_dim), nn.Tanh(),\n        )\n        self.token_queries = nn.Parameter(torch.empty(num_labels, self.attention_dim))\n        nn.init.normal_(self.token_queries, std=0.02)\n        # Initial residual is exactly zero, including for empty dialogues.\n        self.token_evidence_weights = nn.Parameter(torch.zeros(num_labels, self.attention_dim))\n        self.token_evidence_bias = nn.Parameter(torch.zeros(num_labels))\n\n    def _pool_tokens(self, hidden, input_ids, attention_mask):\n        projected = self.token_projection(hidden)\n        valid = attention_mask.bool()\n        for token_id in self.special_token_ids:\n            valid = valid & input_ids.ne(token_id)\n        # Autocast can lower a matmul even when both operands were explicitly\n        # cast to float32; normalize the resulting scores in float32 as well.\n        scores = (projected.float() @ self.token_queries.float().T / math.sqrt(self.attention_dim)).float()\n        scores = scores.masked_fill(~valid.unsqueeze(-1), torch.finfo(scores.dtype).min)\n        weights = torch.softmax(scores, dim=1) * valid.unsqueeze(-1)\n        # All-special/empty windows produce zero evidence, never NaN.\n        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-12)\n        return torch.einsum("wsl,wsd->wld", weights.to(projected.dtype), projected)\n\n    def forward(self, doc_indices, **inputs):\n        representations, token_representations = [], []\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            window_inputs = {key: value[start:start + self.window_batch_size]\n                             for key, value in inputs.items()}\n            hidden = self.encoder(**window_inputs).last_hidden_state\n            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))\n            token_representations.append(self._pool_tokens(\n                hidden, window_inputs["input_ids"], window_inputs["attention_mask"],\n            ))\n        windows = self.dropout(torch.cat(representations))\n        token_windows = torch.cat(token_representations)\n        logits = []\n        for document_index in range(int(doc_indices.max().item()) + 1):\n            selected_indices = doc_indices == document_index\n            selected = windows[selected_indices]\n            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])\n            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)\n            pooled = attention.T @ selected\n            global_logits = (self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias\n            token_pooled = (token_windows[selected_indices] * attention.unsqueeze(-1)).sum(dim=0)\n            evidence = (self.dropout(token_pooled) * self.token_evidence_weights).sum(-1)\n            logits.append(global_logits + evidence + self.token_evidence_bias)\n        return torch.stack(logits)\n\n\ndef _read_checkpoint(path):\n    return torch.load(Path(path), map_location="cpu", weights_only=True, mmap=True)\n\n\ndef _encoder(configuration):\n    values = dict(configuration)\n    model_type = values.pop("model_type")\n    return AutoModel.from_config(AutoConfig.for_model(model_type, **values))\n\n\ndef _special_ids(tokenizer):\n    return sorted({int(value) for name in ("pad_token_id", "cls_token_id", "sep_token_id",\n                                         "bos_token_id", "eos_token_id")\n                   if (value := getattr(tokenizer, name, None)) is not None})\n\n\ndef warm_start(init_checkpoint_path, tokenizer, config):\n    """Create a residual model from a ver4 checkpoint without a model download."""\n    checkpoint = _read_checkpoint(init_checkpoint_path)\n    if checkpoint.get("format_version") != 1 or "model_state" not in checkpoint:\n        raise ValueError("Expected a ver4 fold checkpoint for warm start")\n    if checkpoint.get("format") in (MODEL_FORMAT, LAST_FORMAT):\n        raise ValueError("Warm start requires the original ver4 fold checkpoint")\n    architecture = dict(checkpoint["architecture"])\n    architecture.update(\n        attention_dim=int(config.get("attention_dim", DEFAULTS["attention_dim"])),\n        dropout=float(config.get("dropout", architecture["dropout"])),\n        window_batch_size=int(config.get("window_batch_size", architecture["window_batch_size"])),\n        special_token_ids=_special_ids(tokenizer),\n    )\n    model = DialogueClassifier(_encoder(checkpoint["encoder_config"]), **architecture)\n    missing, unexpected = model.load_state_dict(checkpoint["model_state"], strict=False)\n    expected = {key for key in model.state_dict() if key.startswith("token_")}\n    if set(missing) != expected or unexpected:\n        raise ValueError(f"Warm-start parameter mismatch: missing={missing}, unexpected={unexpected}")\n    metadata = {\n        "encoder_config": checkpoint["encoder_config"], "architecture": architecture,\n        "model_name": checkpoint.get("model_name"),\n        "init_epoch": checkpoint.get("epoch"), "init_seed": checkpoint.get("seed"),\n    }\n    return model, metadata\n\n\ndef _from_saved(checkpoint, config):\n    if checkpoint.get("format") not in (MODEL_FORMAT, LAST_FORMAT) or checkpoint.get("format_version") != 1:\n        raise ValueError("Unsupported ver5 neural checkpoint")\n    architecture = dict(checkpoint["architecture"])\n    if "window_batch_size" in config:\n        architecture["window_batch_size"] = int(config["window_batch_size"])\n    model = DialogueClassifier(_encoder(checkpoint["encoder_config"]), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    return model\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    device = _device(config)\n    checkpoint = _read_checkpoint(checkpoint_path)\n    if checkpoint.get("format") != MODEL_FORMAT:\n        raise ValueError("Inference requires the best ver5 checkpoint, not a resume checkpoint")\n    model = _from_saved(checkpoint, config).to(device)\n    del checkpoint\n    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 8))\n    probabilities = _predict(model, loader, device).astype(np.float32, copy=False)\n    if not np.isfinite(probabilities).all():\n        raise FloatingPointError("Inference produced non-finite probabilities")\n    return probabilities\n\n\ndef _atomic_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(value, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _settings(config):\n    settings = {name: config.get(name, value) for name, value in DEFAULTS.items()}\n    for name in ("warmup_epochs", "finetune_epochs", "patience", "batch_size", "window_batch_size",\n                 "gradient_accumulation_steps", "inference_batch_size", "attention_dim"):\n        value = settings[name]\n        minimum = 0 if name == "warmup_epochs" else 1\n        if isinstance(value, bool) or not isinstance(value, int) or value < minimum:\n            raise ValueError(f"{name} must be an integer >= {minimum}")\n    for name in ("learning_rate", "head_learning_rate", "warmup_head_learning_rate"):\n        if not math.isfinite(float(settings[name])) or float(settings[name]) <= 0:\n            raise ValueError(f"{name} must be finite and positive")\n    if not 0 <= float(settings["dropout"]) < 1 or not 0 <= float(settings["warmup_ratio"]) < 1:\n        raise ValueError("dropout/warmup_ratio must be in [0, 1)")\n    if float(settings["weight_decay"]) < 0 or not math.isfinite(float(settings["pos_weight_power"])):\n        raise ValueError("Invalid weight decay or positive-class weight power")\n    return settings\n\n\ndef _fingerprint(settings, seed, init_checkpoint_path):\n    path = Path(init_checkpoint_path).resolve()\n    data = {"settings": settings, "seed": int(seed), "init_checkpoint": str(path)}\n    # The caller\'s run_signature binds data/code/content; this guards accidental\n    # config changes even when a caller incorrectly reuses the same signature.\n    return hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()\n\n\ndef _rng_state(generator):\n    numpy_state = np.random.get_state()\n    return {\n        "python": random.getstate(),\n        "numpy": {"name": numpy_state[0], "keys": numpy_state[1].tolist(),\n                  "position": int(numpy_state[2]), "has_gauss": int(numpy_state[3]),\n                  "cached_gaussian": float(numpy_state[4])},\n        "torch": torch.get_rng_state(),\n        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n        "loader": generator.get_state(),\n    }\n\n\ndef _restore_rng(state, generator):\n    random.setstate(state["python"])\n    values = state["numpy"]\n    np.random.set_state((values["name"], np.asarray(values["keys"], dtype=np.uint32),\n                         values["position"], values["has_gauss"], values["cached_gaussian"]))\n    torch.set_rng_state(state["torch"])\n    if state["cuda"]:\n        if not torch.cuda.is_available() or len(state["cuda"]) != torch.cuda.device_count():\n            raise ValueError("CUDA RNG device count changed; exact epoch resume is unavailable")\n        torch.cuda.set_rng_state_all(state["cuda"])\n    generator.set_state(state["loader"])\n\n\ndef _phase(epoch, settings):\n    return "warmup" if epoch <= settings["warmup_epochs"] else "finetune"\n\n\ndef _configure_phase(model, phase, settings):\n    for parameter in model.encoder.parameters():\n        parameter.requires_grad_(phase == "finetune")\n    if phase == "finetune" and settings["gradient_checkpointing"]:\n        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    else:\n        model.encoder.gradient_checkpointing_disable()\n    if hasattr(model.encoder.config, "use_cache"):\n        model.encoder.config.use_cache = False\n\n\ndef _phase_optimizer(model, phase, settings, batches):\n    options = dict(settings)\n    if phase == "warmup":\n        options["head_learning_rate"] = settings["warmup_head_learning_rate"]\n    optimizer = _optimizer(model, options)\n    epoch_count = settings["warmup_epochs"] if phase == "warmup" else settings["finetune_epochs"]\n    steps = math.ceil(batches / settings["gradient_accumulation_steps"]) * epoch_count\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(steps * settings["warmup_ratio"]), num_training_steps=steps,\n    )\n    return optimizer, scheduler\n\n\ndef _score(model, loader, device, targets):\n    probabilities = _predict(model, loader, device)\n    if not np.isfinite(probabilities).all():\n        raise FloatingPointError("Stopping predictions contain non-finite values")\n    valid = targets.sum(axis=0) > 0\n    if not valid.any():\n        raise ValueError("Stopping split has no positive labels")\n    return float(average_precision_score(targets[:, valid], probabilities[:, valid], average="macro")), int(valid.sum())\n\n\ndef _check_identity(checkpoint, run_signature, configuration_fingerprint):\n    if checkpoint.get("run_signature") != run_signature:\n        raise ValueError("Checkpoint run signature mismatch; use a new output directory")\n    if checkpoint.get("configuration_fingerprint") != configuration_fingerprint:\n        raise ValueError("Checkpoint config fingerprint mismatch; use a new output directory")\n\n\ndef _best_payload(last):\n    keep = ("format_version", "encoder_config", "architecture", "model_name", "init_epoch", "init_seed",\n            "model_state", "seed", "run_signature", "configuration_fingerprint", "settings")\n    return {**{key: last[key] for key in keep}, "format": MODEL_FORMAT,\n            "epoch": last["best_epoch"], "stop_ap": last["best_stop_ap"]}\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets,\n                tokenizer, config, checkpoint_path, seed, init_checkpoint_path, run_signature):\n    """Adapt one clean fold and resume only from a fully saved epoch boundary.\n\n    The supplied stopping documents must be the original warm-start fold\'s stop\n    split. Base ensemble predictions and calibration/Validation targets are not\n    accepted here. Phase transitions deliberately start a fresh optimizer.\n    """\n    train_targets = _targets(train_targets, train_documents, "train")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    if not isinstance(run_signature, str) or not run_signature:\n        raise ValueError("run_signature must be a nonempty string")\n    settings = _settings(config)\n    fingerprint = _fingerprint(settings, seed, init_checkpoint_path)\n    device = _device(settings)\n    checkpoint_path = Path(checkpoint_path)\n    last_path = checkpoint_path.with_name(checkpoint_path.stem + "_last.ckpt")\n    best_checkpoint = _read_checkpoint(checkpoint_path) if checkpoint_path.exists() else None\n    if best_checkpoint is not None:\n        if best_checkpoint.get("format") != MODEL_FORMAT:\n            raise ValueError("Existing best checkpoint is not a ver5 model")\n        _check_identity(best_checkpoint, run_signature, fingerprint)\n    last = _read_checkpoint(last_path) if last_path.exists() else None\n    if last is not None:\n        if last.get("format") != LAST_FORMAT or last.get("format_version") != 1:\n            raise ValueError("Unsupported epoch-resume checkpoint")\n        _check_identity(last, run_signature, fingerprint)\n        # Last is committed first. A crash before writing best is recoverable\n        # because the newly selected best weights are exactly last.model_state.\n        best_matches = (best_checkpoint is not None and best_checkpoint.get("epoch") == last["best_epoch"]\n                        and best_checkpoint.get("stop_ap") == last["best_stop_ap"])\n        if not best_matches:\n            if last["best_epoch"] != last["completed_epochs"]:\n                raise ValueError("Best checkpoint is missing/inconsistent with the resume record")\n            _atomic_save(_best_payload(last), checkpoint_path)\n        if last["completed"]:\n            return {"best_epoch": last["best_epoch"], "best_stop_ap": last["best_stop_ap"],\n                    "history": last["history"], "completed_epochs": last["completed_epochs"],\n                    "resumed": True, "last_checkpoint": str(last_path)}\n    elif best_checkpoint is not None:\n        raise ValueError("Best checkpoint exists without epoch-resume state; use a new output directory")\n    del best_checkpoint\n\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    if last is None:\n        model, metadata = warm_start(init_checkpoint_path, tokenizer, settings)\n    else:\n        model = _from_saved(last, settings)\n        metadata = {key: last[key] for key in ("encoder_config", "architecture", "model_name", "init_epoch", "init_seed")}\n    model.to(device)\n    train_loader = _loader(train_documents, train_targets, tokenizer, settings["batch_size"], shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, settings["inference_batch_size"])\n    positives = train_targets.sum(axis=0)\n    if (positives == 0).any():\n        raise ValueError("Every training label needs positive examples")\n    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)\n                         ** settings["pos_weight_power"], 1, 5)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    completed_epochs = last["completed_epochs"] if last else 0\n    phase = _phase(completed_epochs + 1, settings)\n    _configure_phase(model, phase, settings)\n    optimizer, scheduler = _phase_optimizer(model, phase, settings, len(train_loader))\n    if last is not None:\n        if phase == last["phase"]:\n            optimizer.load_state_dict(last["optimizer"])\n            scheduler.load_state_dict(last["scheduler"])\n        scaler.load_state_dict(last["scaler"])\n        _restore_rng(last["rng"], train_loader.generator)\n        history = list(last["history"])\n        best_epoch, best_ap, stale = last["best_epoch"], last["best_stop_ap"], last["stale_finetune_epochs"]\n        print(f"완료된 epoch {completed_epochs}부터 재개합니다. 중단된 epoch는 처음부터 다시 실행합니다.", flush=True)\n    else:\n        baseline_started = time.monotonic()\n        best_ap, valid_count = _score(model, stop_loader, device, stop_targets)\n        best_epoch, stale = 0, 0\n        history = [{"epoch": 0, "phase": "baseline", "phase_epoch": 0, "train_loss": None,\n                    "stop_macro_ap": best_ap, "stop_labels_with_positives": valid_count,\n                    "minutes": (time.monotonic() - baseline_started) / 60}]\n        print(f"epoch=0 baseline stop_AP={best_ap:.6f} (warm-start model, no training)", flush=True)\n    resumed = last is not None\n    del last\n\n    def save_state(epoch, done, improved):\n        print(f"epoch={epoch} 재개 상태 저장 중: {last_path.name} (모델·optimizer 포함)", flush=True)\n        state = {"format": LAST_FORMAT, "format_version": 1, **metadata,\n                 "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                 "seed": int(seed), "run_signature": run_signature,\n                 "configuration_fingerprint": fingerprint, "settings": settings,\n                 "completed_epochs": epoch, "phase": phase, "completed": done,\n                 "best_epoch": best_epoch, "best_stop_ap": best_ap,\n                 "stale_finetune_epochs": stale, "history": history,\n                 "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n                 "scaler": scaler.state_dict(), "rng": _rng_state(train_loader.generator)}\n        _atomic_save(state, last_path)\n        if improved:\n            print(f"epoch={epoch} 최적 가중치 저장 중: {checkpoint_path.name}", flush=True)\n            _atomic_save(_best_payload(state), checkpoint_path)\n\n    if not resumed:\n        save_state(0, False, True)\n    total_epochs = settings["warmup_epochs"] + settings["finetune_epochs"]\n    accumulation = settings["gradient_accumulation_steps"]\n    for epoch in range(completed_epochs + 1, total_epochs + 1):\n        next_phase = _phase(epoch, settings)\n        if next_phase != phase:\n            phase = next_phase\n            _configure_phase(model, phase, settings)\n            optimizer, scheduler = _phase_optimizer(model, phase, settings, len(train_loader))\n            stale = 0\n        epoch_started = time.monotonic()\n        model.train()\n        if phase == "warmup":\n            model.encoder.eval()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, document_count = 0.0, 0\n        for step, batch in enumerate(train_loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * train_loader.batch_size,\n                                  len(train_documents) - group_start * train_loader.batch_size)\n            with _amp(device):\n                logits = model(**batch)\n            # Float32 BCE and actual-document weighting include a short final group.\n            loss = criterion(logits.float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += float(loss.detach()) * len(labels)\n            document_count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            if settings["log_every"] > 0 and (step + 1) % settings["log_every"] == 0:\n                training_elapsed = time.monotonic() - epoch_started\n                remaining_training = training_elapsed / (step + 1) * (len(train_loader) - step - 1)\n                print(f"seed={seed} phase={phase} epoch={epoch}/{total_epochs} "\n                      f"batch={step + 1}/{len(train_loader)} loss={loss_sum / document_count:.5f} "\n                      f"minutes={training_elapsed / 60:.1f} "\n                      f"train_ETA_minutes={remaining_training / 60:.1f} (평가·저장 제외)", flush=True)\n        score, valid_count = _score(model, stop_loader, device, stop_targets)\n        row = {"epoch": epoch, "phase": phase,\n               "phase_epoch": epoch if phase == "warmup" else epoch - settings["warmup_epochs"],\n               "train_loss": loss_sum / document_count, "stop_macro_ap": score,\n               "stop_labels_with_positives": valid_count, "minutes": (time.monotonic() - epoch_started) / 60}\n        history.append(row)\n        improved = score > best_ap + 1e-6\n        if improved:\n            best_epoch, best_ap, stale = epoch, score, 0\n        elif phase == "finetune":\n            stale += 1\n        completed_epochs = epoch\n        done = epoch == total_epochs or (phase == "finetune" and stale >= settings["patience"])\n        save_state(epoch, done, improved)\n        print(f"phase={phase} epoch={epoch}/{total_epochs} loss={row[\'train_loss\']:.5f} "\n              f"stop_AP={score:.6f} best_epoch={best_epoch} minutes={row[\'minutes\']:.1f}", flush=True)\n        if done:\n            break\n    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history,\n            "completed_epochs": completed_epochs, "resumed": resumed, "last_checkpoint": str(last_path)}\n', 'calibration.py': '"""Training-only selection between ver4 and a calibrated three-way mixture."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from .ver4_runtime.data import TARGET_SYMPTOMS\nelse:\n    from ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from ver4_runtime.data import TARGET_SYMPTOMS\n\nCANDIDATE_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\n\n\ndef _validated_inputs(y_true, base_neural, sparse, candidate):\n    truth, base, sparse = _inputs(y_true, base_neural, sparse)\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != base.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    return truth, base, sparse, candidate\n\n\ndef _with_zero_candidate(calibrator):\n    return {**calibrator, "candidate_weights": [0.0] * len(TARGET_SYMPTOMS)}\n\n\ndef _fit_mixture(y_true, base_neural, sparse, candidate):\n    """Exact F1 selection per label; equal scores prefer zero candidate weight."""\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    alphas, betas, thresholds = [], [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        best = None\n        for alpha in WEIGHT_GRID:\n            baseline = alpha * base[:, column] + (1.0 - alpha) * sparse[:, column]\n            for beta in CANDIDATE_GRID:\n                probabilities = (1.0 - beta) * baseline + beta * candidate[:, column]\n                threshold, score = _best_threshold(truth[:, column], probabilities)\n                # Keep ver4\'s alpha tie convention after preferring smaller beta.\n                choice = (score, -beta, -abs(alpha - 0.5), alpha, threshold)\n                if best is None or choice > best:\n                    best = choice\n        _, negative_beta, _, alpha, threshold = best\n        alphas.append(float(alpha))\n        betas.append(float(-negative_beta))\n        thresholds.append(float(threshold))\n    return {"neural_weights": alphas, "candidate_weights": betas, "thresholds": thresholds}\n\n\ndef apply_calibrator(base_neural, sparse, candidate, calibrator):\n    """Return mixture probabilities. A zero-beta fallback needs no candidate."""\n    baseline = _apply_baseline(base_neural, sparse, calibrator)\n    betas = np.asarray(calibrator["candidate_weights"], dtype=np.float64)\n    if betas.shape != (len(TARGET_SYMPTOMS),) or not np.isfinite(betas).all():\n        raise ValueError("candidate_weights must contain one finite weight per symptom")\n    if np.any((betas < 0) | (betas > 1)):\n        raise ValueError("candidate_weights must be in [0, 1]")\n    if candidate is None:\n        if np.any(betas != 0):\n            raise ValueError("candidate probabilities are required for nonzero candidate_weights")\n        return baseline\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != baseline.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    return (1.0 - betas) * baseline + betas * candidate\n\n\ndef _metrics(truth, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, predictions, average=None, zero_division=0,\n    )\n    return {\n        symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n        for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n    }\n\n\ndef calibrate_crossfit(y_true, base_neural, sparse, candidate, groups, seed=44, n_splits=5):\n    """Tune on calibration-fold training rows, compare on excluded groups.\n\n    All supplied probabilities must originate from base/candidate models that\n    excluded this entire Training calibration set from model fitting and epoch\n    selection. These arrays must never contain official Validation samples.\n    The global procedure choice uses the resulting cross-fit scores, so its\n    selected score is a selection estimate, not an unbiased performance guarantee.\n    """\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    baseline_predictions = np.zeros_like(truth)\n    candidate_predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        baseline_calibrator = _fit_baseline(truth[train], base[train], sparse[train])\n        mixture_calibrator = _fit_mixture(truth[train], base[train], sparse[train], candidate[train])\n        baseline_predictions[test] = (\n            _apply_baseline(base[test], sparse[test], baseline_calibrator)\n            >= np.asarray(baseline_calibrator["thresholds"])\n        )\n        candidate_predictions[test] = (\n            apply_calibrator(base[test], sparse[test], candidate[test], mixture_calibrator)\n            >= np.asarray(mixture_calibrator["thresholds"])\n        )\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("every calibration sample must be scored exactly once")\n    baseline_score = float(f1_score(truth, baseline_predictions, average="macro", zero_division=0))\n    candidate_score = float(f1_score(truth, candidate_predictions, average="macro", zero_division=0))\n    baseline_final = _with_zero_candidate(_fit_baseline(truth, base, sparse))\n    candidate_final = _fit_mixture(truth, base, sparse, candidate)\n    use_candidate = candidate_score > baseline_score\n    selected = candidate_final if use_candidate else baseline_final\n    selected_predictions = candidate_predictions if use_candidate else baseline_predictions\n    fitted = apply_calibrator(base, sparse, candidate, selected) >= np.asarray(selected["thresholds"])\n    report = {\n        "score_scope": "Training-only calibration. Procedure selection reuses cross-fit scores; selected score is a selection estimate, not an unbiased guarantee or official Validation.",\n        "baseline_crossfit_macro_f1": baseline_score,\n        "candidate_crossfit_macro_f1": candidate_score,\n        "selected_crossfit_macro_f1": candidate_score if use_candidate else baseline_score,\n        "selected_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "selected_name": "candidate_mixture" if use_candidate else "ver4_baseline",\n        "fallback_reason": None if use_candidate else "candidate_crossfit_not_better_than_baseline",\n        "per_symptom_crossfit": {\n            "baseline": _metrics(truth, baseline_predictions),\n            "candidate": _metrics(truth, candidate_predictions),\n            "selected": _metrics(truth, selected_predictions),\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n        "selection_rule": "Use candidate procedure only if its aggregate cross-fit macro F1 strictly exceeds the same-fold ver4 baseline; no per-label gate.",\n    }\n    return selected, report\n', 'pipeline.py': '"""Reuse ver4; fit one complementary model without touching official Validation."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from . import calibration, neural\n    from .ver4_runtime import pipeline as base_pipeline\n    from .ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group, write_json, write_predictions\nelse:\n    import calibration\n    import neural\n    from ver4_runtime import pipeline as base_pipeline\n    from ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group, write_json, write_predictions\n\n\nDEFAULT_CONFIG = {\n    "seed": 42, "init_fold": 1, "warmup_epochs": 1, "finetune_epochs": 3,\n    "patience": 2, "learning_rate": 5e-6, "head_learning_rate": 1e-4,\n    "warmup_head_learning_rate": 3e-4, "attention_dim": 256, "dropout": 0.2,\n    "batch_size": 2, "window_batch_size": 2, "gradient_accumulation_steps": 16,\n    "inference_batch_size": 8, "gradient_checkpointing": True,\n    "layerwise_lr_decay": 0.95, "weight_decay": 0.01, "pos_weight_power": 0.5,\n    "warmup_ratio": 0.1, "log_every": 100, "device": "cuda", "require_cuda": True,\n}\nRUNTIME_FILES = (\n    "__init__.py", "pipeline.py", "neural.py", "calibration.py", "train.py", "inference.py",\n    "requirements.txt", "README.md", "RESEARCH.md",\n)\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _atomic_torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _release():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _versions():\n    return {name: importlib.metadata.version(name) for name in\n            ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n\n\ndef _check_versions(base_manifest):\n    current = _versions()\n    # sklearn objects in sparse.joblib require the original compatible environment.\n    for name in ("transformers", "numpy", "scikit-learn"):\n        saved = base_manifest.get("versions", {}).get(name)\n        if saved and current[name] != saved:\n            raise RuntimeError(f"{name}: ver4 저장 버전={saved}, 현재={current[name]}. requirements.txt를 설치하세요.")\n    return current\n\n\ndef _read_base(train_label_dir, base_dir, init_fold):\n    """Verify the exact original training run before trusting cached predictions."""\n    checkpoint = base_dir / "mission3_ver4_model.ckpt"\n    manifest = base_pipeline.load_manifest(checkpoint)\n    _check_versions(manifest)\n    print("기존 ver4 확인 완료. 서울 Training JSON을 읽습니다...", flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    print(f"Training {len(paths):,}개 로드 완료. 원래 데이터·분할·보정 캐시를 검증합니다...", flush=True)\n    fingerprint = base_pipeline._fingerprint(paths, texts, targets, manifest["config"])\n    if fingerprint != manifest.get("signature"):\n        raise ValueError("ver4와 Training 데이터/전처리/코드가 다릅니다. 원래 Training과 ver4 결과 묶음을 사용하세요.")\n    splits = json.loads((base_dir / "splits.json").read_text(encoding="utf-8"))\n    lookup = {p.name: i for i, p in enumerate(paths)}\n    all_names = set(lookup)\n    group_ids = np.asarray([text_group(text) for text in texts])\n\n    def indices(names, purpose):\n        if not names or len(names) != len(set(names)) or not set(names) <= all_names:\n            raise ValueError(f"Invalid saved {purpose} filenames")\n        return np.asarray([lookup[name] for name in names], dtype=np.int64)\n\n    cal = indices(splits["calibration"], "calibration")\n    if len(splits["folds"]) != len(manifest["folds"]) or not 1 <= init_fold <= len(splits["folds"]):\n        raise ValueError("init_fold / saved folds mismatch")\n    fold_indices = []\n    for number, fold in enumerate(splits["folds"], 1):\n        fit, stop = indices(fold["fit"], "fit"), indices(fold["stop"], "stop")\n        sets = [set(fit), set(stop), set(cal)]\n        groups = [set(group_ids[x]) for x in (fit, stop, cal)]\n        if any(sets[i] & sets[j] or groups[i] & groups[j] for i, j in ((0, 1), (0, 2), (1, 2))):\n            raise ValueError(f"fold {number}: sample or text-group leakage")\n        if set.union(*sets) != set(range(len(paths))):\n            raise ValueError(f"fold {number}: saved partitions do not cover Training")\n        fold_indices.append((fit, stop))\n    # Verify the stored fold identities against the ORIGINAL deterministic run.\n    # We still use the stored indices below; a new split after warm-start is unsafe.\n    _, expected_cal, expected_folds, _ = base_pipeline.make_splits(\n        texts, targets, manifest["config"]["calibration_size"],\n        manifest["config"]["n_folds"], manifest["config"]["seed"],\n    )\n    if not np.array_equal(cal, expected_cal) or any(\n        not np.array_equal(actual, expected)\n        for pair, reference in zip(fold_indices, expected_folds)\n        for actual, expected in zip(pair, reference)\n    ):\n        raise ValueError("Saved fold identity differs from the original ver4 run; warm-start would leak.")\n    for number, name in enumerate(manifest["folds"], 1):\n        saved = torch.load(base_dir / name, map_location="cpu", weights_only=True, mmap=True)\n        if (saved.get("format_version") != 1\n                or saved.get("seed") != manifest["config"]["seed"] + number * 1009\n                or saved.get("architecture", {}).get("num_labels") != len(TARGET_SYMPTOMS)):\n            raise ValueError(f"Saved checkpoint fold identity mismatch: {name}")\n        del saved\n    with np.load(base_dir / "calibration_predictions.npz", allow_pickle=False) as cache:\n        names = cache["filenames"].tolist()\n        if len(set(names)) != len(names) or set(names) != set(splits["calibration"]):\n            raise ValueError("ver4 calibration cache filenames mismatch")\n        order = {name: i for i, name in enumerate(names)}\n        aligned = [order[paths[i].name] for i in cal]\n        raw_neural = np.array(cache["neural"][aligned], dtype=np.float64)\n        raw_sparse = np.array(cache["sparse"][aligned], dtype=np.float64)\n        if not np.array_equal(cache["targets"][aligned], targets[cal]):\n            raise ValueError("ver4 calibration cache targets mismatch")\n    for values in (raw_neural, raw_sparse):\n        if values.shape != (len(cal), len(TARGET_SYMPTOMS)) or not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):\n            raise ValueError("Invalid ver4 raw calibration probabilities")\n    fit, stop = fold_indices[init_fold - 1]\n    return manifest, paths, texts, targets, fit, stop, cal, group_ids, raw_neural, raw_sparse\n\n\ndef _run_signature(config, base_manifest, base_dir, init_checkpoint):\n    source_dir = Path(__file__).parent\n    code = hashlib.sha256()\n    own_code = [source_dir / name for name in RUNTIME_FILES if name.endswith(".py")]\n    for path in sorted([*own_code, *source_dir.joinpath("ver4_runtime").glob("*.py")]):\n        code.update(str(path.relative_to(source_dir)).encode())\n        code.update(path.read_bytes())\n    assets = [base_dir / name for name in [*base_manifest["folds"], base_manifest["sparse"], "mission3_ver4_model.ckpt"]]\n    assets += [path for path in (base_dir / base_manifest["tokenizer"]).rglob("*") if path.is_file()]\n    asset_hashes = {}\n    for path in sorted(assets):\n        name = str(path.relative_to(base_dir))\n        print("기본 모델 무결성 확인:", name, flush=True)\n        asset_hashes[name] = _sha256(path)\n    value = {\n        "config": config, "base_signature": base_manifest["signature"],\n        "initial_checkpoint_sha256": asset_hashes[str(init_checkpoint.relative_to(base_dir))],\n        "base_artifact_sha256": asset_hashes,\n        "calibration_cache_sha256": _sha256(base_dir / "calibration_predictions.npz"),\n        "splits_sha256": _sha256(base_dir / "splits.json"), "code_sha256": code.hexdigest(),\n        "versions": _versions(),\n    }\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest(), value\n\n\ndef _copy_file(source, target, expected_hash):\n    """An interrupted copy never appears under its final filename."""\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        if target.stat().st_size != source.stat().st_size or _sha256(target) != expected_hash:\n            raise ValueError(f"기존 묶음 파일 내용이 다릅니다: {target}")\n        return\n    temporary = target.with_name(target.name + ".tmp")\n    digest = hashlib.sha256()\n    with source.open("rb") as reader, temporary.open("wb") as writer:\n        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n            writer.write(chunk)\n            digest.update(chunk)\n    if digest.hexdigest() != expected_hash:\n        temporary.unlink()\n        raise ValueError(f"복사 중 기본 모델 파일이 변경되었습니다: {source}")\n    temporary.replace(target)\n\n\ndef _export_base(base_dir, output_dir, manifest, asset_hashes):\n    destination = output_dir / "base"\n    destination.mkdir(exist_ok=True)\n    # Copy to make the resulting ver5 folder portable without its old sibling.\n    for name in [*manifest["folds"], manifest["sparse"], "mission3_ver4_model.ckpt"]:\n        print("Bundle:", name, flush=True)\n        _copy_file(base_dir / name, destination / name, asset_hashes[name])\n    tokenizer_dir = base_dir / manifest["tokenizer"]\n    for path in tokenizer_dir.rglob("*"):\n        if path.is_file():\n            _copy_file(path, destination / manifest["tokenizer"] / path.relative_to(tokenizer_dir),\n                       asset_hashes[str(path.relative_to(base_dir))])\n    copied = base_pipeline.load_manifest(destination / "mission3_ver4_model.ckpt")\n    if copied["signature"] != manifest["signature"]:\n        raise ValueError("Portable base bundle signature mismatch")\n\n\ndef _export_code(output_dir):\n    source = Path(__file__).parent\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in RUNTIME_FILES:\n        if (source / name).is_file():\n            shutil.copyfile(source / name, submission / name)\n    for path in source.joinpath("ver4_runtime").iterdir():\n        if path.is_file():\n            destination = submission / "ver4_runtime" / path.name\n            destination.parent.mkdir(exist_ok=True)\n            shutil.copyfile(path, destination)\n\n\ndef train_pipeline(train_label_dir, base_dir, output_dir, config=None):\n    """Only TL_서울_구급 is accepted. Validation is deliberately not an argument."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    train_label_dir, base_dir, output_dir = (Path(x).expanduser().resolve() for x in\n                                          (train_label_dir, base_dir, output_dir))\n    if train_label_dir.name != "TL_서울_구급" or any("validation" in part.lower() for part in train_label_dir.parts):\n        raise ValueError("학습 입력은 서울 Training/TL_서울_구급 폴더여야 합니다.")\n    if output_dir == base_dir or output_dir.is_relative_to(base_dir) or base_dir.is_relative_to(output_dir):\n        raise ValueError("ver5 출력은 기존 ver4와 별도 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("ver5 학습에는 GPU 런타임을 선택하세요.")\n    (base_manifest, paths, texts, targets, fit, stop, cal, groups,\n     base_probs, sparse_probs) = _read_base(train_label_dir, base_dir, int(config["init_fold"]))\n    init_checkpoint = base_dir / base_manifest["folds"][int(config["init_fold"]) - 1]\n    signature, provenance = _run_signature(config, base_manifest, base_dir, init_checkpoint)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8")).get("signature") != signature:\n        raise ValueError("ver5 출력 폴더의 데이터/설정/초기 가중치/코드가 다릅니다. 새 OUTPUT_DIR를 지정하세요.")\n    if not run_path.exists() and any(output_dir.glob("*.ckpt")):\n        raise ValueError("run.json 없는 기존 checkpoint가 있습니다. 새 OUTPUT_DIR를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "provenance": provenance})\n    manifest_path = output_dir / "mission3_ver5_model.ckpt"\n    report_path = output_dir / "mission3_ver5_metrics.json"\n    if manifest_path.is_file() and report_path.is_file():\n        saved = load_manifest(manifest_path)\n        if saved["signature"] != signature:\n            raise ValueError("Final ver5 signature mismatch")\n        print("완료된 ver5 재사용: 추가 학습·보정 탐색을 건너뜁니다.", flush=True)\n        return manifest_path, json.loads(report_path.read_text(encoding="utf-8"))\n\n    write_json(output_dir / "splits.json", {\n        "fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop],\n        "calibration": [paths[i].name for i in cal], "init_fold": config["init_fold"],\n        "source": "Exact stored ver4 fold; no resplitting after warm-start",\n    })\n    print(f"ver4 재사용 | 새 모델 fit={len(fit):,}, stop={len(stop):,}, calibration={len(cal):,}", flush=True)\n    print(f"새 학습: frozen {config[\'warmup_epochs\']} + fine-tune 최대 {config[\'finetune_epochs\']} epoch", flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(base_dir / base_manifest["tokenizer"], use_fast=True, local_files_only=True)\n    neural_config = {**config, "max_length": base_manifest["config"]["max_length"],\n                     "stride": base_manifest["config"]["stride"]}\n    documents = neural.tokenize_dialogues(texts, tokenizer, neural_config["max_length"], neural_config["stride"])\n    candidate_path, done_path = output_dir / "candidate_best.ckpt", output_dir / "candidate_done.json"\n    if candidate_path.is_file() and done_path.is_file():\n        record = json.loads(done_path.read_text(encoding="utf-8"))\n        if record.get("signature") != signature:\n            raise ValueError("Candidate completion signature mismatch")\n        history = record["training"]\n        print("완료된 추가 모델 재사용", flush=True)\n    else:\n        history = neural.train_model(\n            [documents[i] for i in fit], targets[fit], [documents[i] for i in stop], targets[stop],\n            tokenizer, neural_config, candidate_path, seed=int(config["seed"]) + 5003,\n            init_checkpoint_path=init_checkpoint, run_signature=signature,\n        )\n        write_json(done_path, {"signature": signature, "training": history})\n    _release()\n    candidate_cache = output_dir / "candidate_calibration.npz"\n    if candidate_cache.is_file():\n        with np.load(candidate_cache, allow_pickle=False) as cache:\n            if cache["signature"].item() != signature or cache["filenames"].tolist() != [paths[i].name for i in cal]:\n                raise ValueError("Candidate calibration cache mismatch")\n            candidate_probs = np.array(cache["probabilities"])\n    else:\n        candidate_probs = neural.predict_checkpoint(candidate_path, [documents[i] for i in cal], tokenizer, neural_config)\n        temporary = candidate_cache.with_name(candidate_cache.name + ".tmp")\n        with temporary.open("wb") as stream:\n            np.savez_compressed(stream, probabilities=candidate_probs,\n                                filenames=np.asarray([paths[i].name for i in cal]), signature=np.asarray(signature))\n        temporary.replace(candidate_cache)\n    _release()\n    # RAW ver4 predictions enter cross-fit; its full-calibration thresholds never do.\n    calibrator, report = calibration.calibrate_crossfit(\n        targets[cal], base_probs, sparse_probs, candidate_probs, groups[cal],\n        seed=int(base_manifest["config"]["seed"]) + 2,\n        n_splits=int(base_manifest["config"]["calibration_folds"]),\n    )\n    report.update({\n        "signature": signature, "base_signature": base_manifest["signature"],\n        "training_samples": len(paths), "candidate_fit_samples": len(fit), "candidate_stop_samples": len(stop),\n        "candidate_training": history, "calibrator": calibrator, "config": config,\n        "versions": _versions(), "validation_used_for_training_or_selection": False,\n        "new_models_trained": 1, "base_models_retrained": 0,\n    })\n    _export_base(base_dir, output_dir, base_manifest, provenance["base_artifact_sha256"])\n    _export_code(output_dir)\n    manifest = {\n        "format": "mission3_ver5", "version": 1, "signature": signature,\n        "target_symptoms": list(TARGET_SYMPTOMS), "base_manifest": "base/mission3_ver4_model.ckpt",\n        "base_signature": base_manifest["signature"], "candidate": "candidate_best.ckpt",\n        "config": neural_config, "calibrator": calibrator, "versions": _versions(),\n        "selected_name": report["selected_name"],\n    }\n    write_json(report_path, report)\n    _atomic_torch_save(manifest_path, manifest)\n    print(f"보정 교차평가: ver4={report[\'baseline_crossfit_macro_f1\']:.6f}, "\n          f"ver5 후보={report[\'candidate_crossfit_macro_f1\']:.6f}, 선택={report[\'selected_name\']}", flush=True)\n    print("이 점수는 Training 내부 선택용이며 공식 Validation 점수가 아닙니다.", flush=True)\n    return manifest_path, report\n\n\ndef load_manifest(checkpoint_path):\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver5" or manifest.get("version") != 1:\n        raise ValueError("mission3_ver5_model.ckpt를 지정하세요.")\n    if manifest.get("target_symptoms") != list(TARGET_SYMPTOMS):\n        raise ValueError("ver5 target label order mismatch")\n    root = checkpoint_path.parent\n    for name in (manifest["base_manifest"], manifest["candidate"]):\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.is_file():\n            raise FileNotFoundError(f"ver5 결과 폴더 전체를 함께 보관하세요: {name}")\n    base = base_pipeline.load_manifest(root / manifest["base_manifest"])\n    if base.get("signature") != manifest.get("base_signature"):\n        raise ValueError("ver5 references a different ver4 base")\n    return manifest\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    base_manifest = base_pipeline.load_manifest(root / manifest["base_manifest"])\n    _check_versions(base_manifest)\n    base_root = (root / manifest["base_manifest"]).parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = int(batch_size)\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(base_root / base_manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = neural.tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    alpha = np.asarray(manifest["calibrator"]["neural_weights"])\n    beta = np.asarray(manifest["calibrator"]["candidate_weights"])\n    base_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    sparse_probs = np.zeros_like(base_probs)\n    candidate_probs = None\n    if np.any((1 - beta) * alpha > 0):\n        base_config = {**base_manifest["config"], "device": config["device"],\n                       "inference_batch_size": config["inference_batch_size"]}\n        # Keep ver4\'s original window batching for its numerical inference path.\n        for name in base_manifest["folds"]:\n            print("Predict base:", name, flush=True)\n            base_probs += base_pipeline.predict_checkpoint(base_root / name, documents, tokenizer, base_config)\n            _release()\n        base_probs /= len(base_manifest["folds"])\n    if np.any((1 - beta) * (1 - alpha) > 0):\n        sparse_probs = base_pipeline.predict_sparse(joblib.load(base_root / base_manifest["sparse"]), texts)\n    if np.any(beta > 0):\n        print("Predict: candidate_best.ckpt", flush=True)\n        candidate_probs = neural.predict_checkpoint(root / manifest["candidate"], documents, tokenizer, config)\n        _release()\n    probabilities = calibration.apply_calibrator(base_probs, sparse_probs, candidate_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\nevaluate_csv = base_pipeline.evaluate_csv\n', 'train.py': '"""Train the complementary model and select calibration using Training only."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train_label_dir", required=True)\n    parser.add_argument("--base_dir", required=True)\n    parser.add_argument("--output_dir", required=True)\n    parser.add_argument("--config_json")\n    parser.add_argument("--auto_batch", action="store_true")\n    args = parser.parse_args()\n    config = json.loads(Path(args.config_json).read_text(encoding="utf-8")) if args.config_json else {}\n    if args.auto_batch:\n        run_path = Path(args.output_dir) / "run.json"\n        if run_path.is_file():\n            # Keep the original batch schedule when resuming on another GPU.\n            saved = json.loads(run_path.read_text(encoding="utf-8"))["config"]\n            for key in ("batch_size", "window_batch_size", "gradient_accumulation_steps"):\n                config.setdefault(key, saved[key])\n            print("이전 실행의 batch 설정을 유지하여 재개합니다.", flush=True)\n        elif torch.cuda.is_available():\n            memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\n            batch = 8 if memory_gb >= 35 else 4 if memory_gb >= 22 else 2\n            config.setdefault("batch_size", batch)\n            config.setdefault("window_batch_size", batch)\n            config.setdefault("gradient_accumulation_steps", 32 // config["batch_size"])\n    print("ver5 추가 학습 설정:", json.dumps(config, ensure_ascii=False), flush=True)\n    checkpoint, report = train_pipeline(args.train_label_dir, args.base_dir, args.output_dir, config)\n    print("Checkpoint:", checkpoint, flush=True)\n    print("선택:", report["selected_name"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""One-command Mission 3 inference; only transcript text is a model input."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import evaluate_csv, predict_pipeline\n    from .ver4_runtime.data import write_json\nelse:\n    from pipeline import evaluate_csv, predict_pipeline\n    from ver4_runtime.data import write_json\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", default="", help="Accepted for the competition interface; text task does not read audio.")\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", choices=("cuda", "cpu"))\n    parser.add_argument("--batch_size", type=int)\n    parser.add_argument("--evaluate", action="store_true", help="Evaluate only after CSV is saved and all targets exist.")\n    args = parser.parse_args()\n    paths, _, _ = predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size)\n    if args.evaluate:\n        target_count = sum("symptom" in json.loads(path.read_text(encoding="utf-8-sig")) for path in paths)\n        if target_count == len(paths):\n            report = evaluate_csv(args.label_dir, args.output)\n            write_json(Path(args.output).with_name("validation_metrics.json"), report)\n            print(f"Final evaluation Macro F1: {report[\'macro_f1\']:.6f}", flush=True)\n            for name, values in report["per_symptom"].items():\n                print(f"  {name}: {values[\'f1\']:.6f}", flush=True)\n        else:\n            print(f"정답 포함 JSON {target_count}/{len(paths)}: 예측 완료, 평가는 생략합니다.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'README.md': '# Mission 3 ver5\n\nver4의 학습된 다섯 모델을 재사용하고, **증상별 token attention을 추가한 모델 한 개**를 학습합니다.\n논문 3개 PDF는 서로 다른 연구 2편이며 전체 검토 결과와 적용 근거는 [RESEARCH.md](RESEARCH.md)에 있습니다.\n새 구조의 성능 향상은 전체 GPU 실행 전에는 확인되지 않았습니다.\n\n## 실행\n\n`src/mission3/mission3_ver5.ipynb`를 Colab에 올리고 GPU 런타임에서 모두 실행하세요.\n노트북에는 실행 코드가 포함되어 있습니다. 기존 ver4 결과 폴더 전체와 서울 Training JSON이 필요합니다.\n기본 경로:\n\n- 모델: `MyDrive/DCC_AIdeas/results/mission3/ver4`\n- 데이터: `MyDrive/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급`\n  또는 `MyDrive/DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급`\n- 새 결과: `MyDrive/DCC_AIdeas/results/mission3/ver5`\n\nver4 폴더에는 가중치 외에도 `splits.json`, `calibration_predictions.npz`, tokenizer, sparse 모델,\n`mission3_ver4_model.ckpt`가 필요합니다. 원래 Training과 source fingerprint가 일치하는지 확인한 뒤 재사용합니다.\n\n## 모델과 비교\n\n1. 저장된 fold 1의 **원래 fit/stop**을 복원하고 fold 1 최적 가중치에서 시작합니다.\n2. 기존 CLS/mean/window head를 유지하고, padding·특수 토큰을 제외한 증상별 token attention의 결과를 더합니다.\n   추가 출력의 초기값은 0이므로 학습 전에는 기존 fold 1과 같은 예측을 냅니다.\n3. encoder 고정 상태로 head를 1 epoch 학습하고, 전체 encoder를 최대 3 epoch 미세조정합니다.\n   후보 단독의 stop Macro AP로 최적 epoch를 선택합니다. 초기 모델(epoch 0)도 보관 후보입니다.\n4. ver4의 **원시** 5-model 평균 확률·TF-IDF 확률과 새 후보 확률을 Training 보정 집합에서 비교합니다.\n   각 교차평가 fold마다 혼합 비율과 임계값을 해당 fit 부분에서만 선택합니다.\n5. 후보 절차의 교차평가 Macro F1이 더 높을 때만 채택하며, 동률·하락이면 기존 ver4 보정으로 돌아갑니다.\n   이 선택 점수는 비편향 성능 보장이 아닙니다. 공식 Validation은 마지막 CSV 생성 후에만 평가합니다.\n\n다섯 기존 모델을 다시 학습하거나 대규모 학습률 탐색을 하지 않습니다. 혼합 비율 비교는 캐시된 확률에 대한\nCPU 계산입니다. 선택된 비율이 0인 성분은 최종 추론에서 생략합니다.\n\n## 시간과 재개\n\n이전 A100 기록은 22 epoch, 학습+stop 평가 합 630.28분입니다. ver5는 frozen 1 + fine-tune 최대 3 epoch로,\n**전체 2~4시간을 계획용으로 예상**합니다. 아직 실측이 아니며 GPU·Drive I/O·저장/추론 시간에 따라 달라집니다.\n첫 실행의 무결성 검사와 독립 제출 묶음 복사는 기존 6.4GiB 가중치를 읽으므로 시간이 필요합니다.\n최종 가중치·재개 상태·기본 모델 사본을 위해 기존 ver4 외에 약 12GiB 이상의 추가 저장 공간을 준비하세요.\n\nColab에서는 GPU 메모리에 따라 batch/window를 8/8, 4/4, 2/2로 정하고 accumulation으로 유효 batch 32를 유지합니다.\n메모리 부족이 발생하면 새 출력 폴더에서 batch/window를 줄여 실행하세요. 자동 설정이 모든 GPU에서의 성공을 보장하지는 않습니다.\n\n- `candidate_best.ckpt`: stop AP가 가장 좋았던 후보 모델.\n- `candidate_best_last.ckpt`: 마지막 **완료 epoch**의 optimizer/scheduler/scaler/RNG를 포함한 재개 상태.\n- 중간에 중단하면 완료된 epoch까지 복원하고, 미완료 epoch는 다시 시작합니다.\n- 같은 데이터·설정·코드로 재실행하면 완료된 후보 학습과 보정 캐시를 재사용합니다.\n- 기존 ver4에서 시작하는 첫 학습은 새 optimizer로 수행하는 추가 학습이며, ver4 학습의 정확한 재개가 아닙니다.\n\n## 제출\n\n완료된 `results/mission3/ver5` 폴더는 `base/`에 기존 가중치와 tokenizer를 포함하므로 별도로 옮겨도 추론할 수 있습니다.\n`mission3_ver5_model.ckpt` 하나만 복사하지 마세요. 재개용 `*_last.ckpt`는 추론에 필요하지 않습니다.\n\n```bash\npython results/mission3/ver5/submission/inference.py \\\n  --audio_dir "WAV 폴더" --label_dir "JSON 폴더" \\\n  --ckpt_path results/mission3/ver5/mission3_ver5_model.ckpt \\\n  --output mission3.csv\n```\n\n학습 CLI:\n\n```bash\npython src/mission3/mission3_ver5/train.py \\\n  --train_label_dir "Training/2.라벨링데이터/TL_서울_구급" \\\n  --base_dir results/mission3/ver4 --output_dir results/mission3/ver5 --auto_batch\n```\n\n노트북 재생성: `python src/mission3/mission3_ver5/build_notebook.py`.\n최종 CSV는 `label file name,symptom`, 증상은 지정된 9개 문자열 중 0~9개입니다.\n학습 로그가 포함된 노트북을 저장하고, 대회가 요구하는 결과 정리 PPT는 별도로 준비하세요.\n', 'RESEARCH.md': '# Mission 3 ver5: 논문 근거와 설계 판단\n\nver5는 ver4의 학습 결과를 재사용하고, 증상별 토큰 attention을 추가한 후보 하나를 검증하는 설계다.\n성능 향상은 아직 실측하지 않았다. Training 내부 비교에서 개선이 확인되지 않으면 ver4를 유지한다.\n판단 기준은 `docs/0820_DCC 대학부_예선_출제문제.pdf`의 Mission 3 조건이다.\n공식 Validation은 최종 평가에만 사용하며, 서울 Training의 전사 text만 학습·보정에 사용한다.\n\n## 1. 제공 파일 확인\n\nPDF는 3개지만 서로 다른 논문은 2편이다. 아래 쪽수는 각 PDF의 첫 장을 1쪽으로 센다.\n\n| 로컬 파일 | 논문 / 확인 결과 |\n|---|---|\n| `1-s2.0-S2405844022009719-main.pdf` | Heliyon 2022 논문, 12쪽 |\n| `Automatic-symptoms-identification-from-a-massive-volume-of-unstruct_2022_Hel.pdf` | 위 논문의 중복 사본: 12쪽 본문과 이미지 41개가 같음 |\n| `2403.06734v1.pdf` | CognitiveEMS 논문 v1, 13쪽, 2024-03-11 |\n\nHeliyon 사본들의 파일 크기와 SHA-256은 다르므로 파일 자체가 바이트 단위로 같다는 뜻은 아니다.\n내용이 같은 두 사본을 서로 독립적인 성능 향상 근거로 계산하지 않는다.\n원문: [Heliyon DOI](https://doi.org/10.1016/j.heliyon.2022.e09683), [CognitiveEMS arXiv v1](https://arxiv.org/abs/2403.06734v1).\n\n## 2. Heliyon: 대규모 상담에서 증상 식별\n\n제목은 *Automatic symptoms identification from a massive volume of unstructured medical consultations using deep neural and BERT models*이다.\n아랍어 원격의료 상담에서 ICD-10 기반 증상을 추천하는 연구다. 한국어 119 신고 분류 실험이 아니다.\n\n- 4·6쪽: AraBERT를 특징 추출기로 사용하며 BERT의 모든 층을 동결한다.\n- 추출한 특징에 BiLSTM과 sigmoid 분류기를 연결한다. 분류 대상은 2,348개 증상이다.\n- 임베딩 단계의 상담은 3,310,996개이며, 분류 실험의 최종 자료는 501,004개다.\n- 희소 label 제거, label별 최대 5,000개 제한, 중복 제거 등 데이터 구성이 현재 과제와 다르다.\n- 6쪽의 전체 token embedding 설명과 7쪽의 CLS만 사용하는 설명 사이에는 모호함이 있다.\n- 따라서 논문이 특정 token pooling 구현을 일관되게 입증했다고 해석하지 않는다.\n- 7–8쪽: 80/10/10 분할, Adam, batch 128을 사용한다. Table 2는 AraBERT 계열 10 epoch, BiLSTM 비교 모델 50 epoch를 제시한다.\n- 9–10쪽: top-5 증상 추천을 평가하며, 전체 hit rate는 AraBERTv2 0.564, v1 0.537, Word2Vec 0.535다.\n- 전문가가 검토한 500개에서 recall 0.706, precision 0.233, 최소 한 개 정답을 찾은 비율 0.711을 보고한다.\n- 이 수치들은 9개 고정 증상의 Macro F1과 정의·label 수·데이터가 모두 달라 직접 비교할 수 없다.\n\n2쪽은 의사 부착 label의 불완전성을 논의한다. 그러나 자동 정답 수정이나 특정 noise-robust loss를 검증한 논문은 아니다.\n이 언급을 근거로 제공 정답을 임의 변경하거나, 모델의 오탐을 모두 annotation 오류로 처리하지 않는다.\n\n**적용:** 이미 학습된 표현을 보존한 채 새 분류 부분을 먼저 적응시키는 동결 학습의 취지를 사용한다.\nver5의 encoder 동결 1 epoch는 새 attention head 준비 단계다. 이후 전체 encoder를 미세조정한다.\n이는 논문의 영구 동결 AraBERT+BiLSTM을 재현한 것이 아니라 현재 모델에 맞춘 단계적 학습이다.\nBiLSTM, 아랍어 모델, top-5 강제 추천, 희소 증상 삭제는 도입하지 않는다.\n\n## 3. CognitiveEMS: 실시간 응급의료 보조 시스템\n\n제목은 *Real-Time Multimodal Cognitive Assistant for Emergency Medical Services*다.\n전체 시스템은 영어 음성 인식, EMS 프로토콜 선택, 영상의 처치 행동 인식으로 구성된다.\n현재 과제에 직접 참고할 부분은 6–7쪽 §III.C의 프로토콜 다중 label 분류기다.\n\n- TinyClinicalBERT의 마지막 token hidden state로 전사 표현을 만든다.\n- 외부 ODEMSA 지침에서 프로토콜·증상·약물·처치를 추출해 이종 지식 그래프를 구성한다.\n- GatorTron으로 초기 node embedding을 만들고, 1층 HGT로 관계를 반영한다.\n- HLA는 각 프로토콜이 전사의 각 token에 다른 가중치를 주어 관련 표현을 모은다.\n- sigmoid와 BCE를 사용한다. 지식 융합은 기본 15M 모델에 약 0.9M parameter를 추가한다.\n- Group-wise 학습은 성인/소아의 유사 프로토콜을 합친 후 추론 시 나이 규칙으로 복원한다.\n- 7–9쪽: 영어 ePCR 4,417개, 프로토콜 43개를 사용하며 70:30 분할 및 3-fold 평가 평균을 설명한다.\n- 9쪽: HGT hidden 256, dropout 0.3, weight decay 1e-5와 조기 종료를 사용한다.\n- 학습 batch 4–32, learning rate 1e-6–1e-3 탐색을 수행했으나 총 학습 시간은 제시하지 않는다.\n\n10쪽 Table III는 각 추가 요소를 제거했을 때의 결과를 비교한다.\n\n| 모델 구성 | Micro F1 | Macro F1 | Acc@3 |\n|---|---:|---:|---:|\n| TinyClinicalBERT | 0.673 | 0.171 | 0.807 |\n| + group-wise 학습 | 0.709 | 0.277 | 0.836 |\n| + 지식 융합 | 0.743 | 0.395 | 0.898 |\n| 두 요소 모두 | 0.756 | 0.499 | 0.904 |\n\n지식 융합의 이점은 이 연구에서 관찰됐지만, 한국어 증상 분류의 향상폭을 예측하는 값은 아니다.\n11쪽의 서버 0.01초·edge 0.14초는 프로토콜 **추론** 지연이다. A100 학습 시간으로 해석하지 않는다.\n8쪽은 일부 자체 수집 multimodal 시나리오의 전사 정보 부족으로 프로토콜 예측이 잘되지 않았다고 명시한다.\n5–6쪽의 GPT-4·ElevenLabs 합성 자료는 ASR용이며, 전사가 제공되는 본 과제에 추가하지 않는다.\n공식 [EMS-Pipeline](https://github.com/UVA-DSA/EMS-Pipeline)의 분류 구현 링크는 후속 [DKEC](https://github.com/UVA-DSA/DKEC)로 연결된다.\n현재 공개 저장소 전체를 v1 PDF와 동일한 구현이라고 가정하지 않는다.\n\n**적용:** 한국어 encoder를 유지하면서 각 증상이 token 증거를 선택하게 하는 attention을 추가한다.\n외부 지식 그래프, 영어 encoder, 처치/나이 규칙, 음성·영상 처리는 가져오지 않는다.\n구토와 오심처럼 관련된 증상을 합치는 것은 논문의 성인/소아 프로토콜 통합과 다르므로 수행하지 않는다.\n\n## 4. ver4에서 확인한 개선 지점\n\nver4는 token을 CLS+평균으로 압축한 뒤 window 사이에서 증상별 attention을 계산한다.\n한 window짜리 통화에서는 attention 가중치가 항상 1이므로, 그 attention이 특정 token을 고르지 못한다.\nver5는 기존 전역·window 표현에 masked token-wise label attention을 잔차 형태로 더한다.\n새 잔차 출력은 0으로 초기화하여 평가 모드의 최초 예측이 저장된 ver4 fold와 같게 시작한다.\npadding과 special token을 attention 증거에서 제외하고, 긴 통화의 모든 window는 계속 처리한다.\n원래 BCE, 9개 증상 정의, 전사 전처리, 빈 증상 결과 허용은 유지한다.\n\n이 설계는 논문에서 영감을 받은 변경이다. 외부 지식 융합까지 포함한 CognitiveEMS 재현은 아니다.\n새 head의 attention 연산은 encoder를 하나 더 실행하는 것보다 작지만 실제 시간 증가는 측정해야 한다.\n별도 BiLSTM, ASL, label 자동 재작성, 여러 backbone·learning rate의 광범위 탐색은 이번 범위에 넣지 않는다.\n\n## 5. 학습·선택·평가 절차\n\n1. ver4의 실제 모델 5개, tokenizer, TF-IDF, 분할 목록, 원시 calibration 예측 확률을 읽는다.\n2. 기존 fold 1의 최적 가중치에서 후보 하나를 시작하며 동일한 fit/stopping 자료를 사용한다.\n3. 최대 1 epoch는 encoder를 동결해 새 head를 준비하고, 최대 3 epoch는 전체를 미세조정한다.\n4. stopping 평가로 후보 가중치를 고른다. calibration 정답은 encoder 학습과 epoch 선택에 쓰지 않는다.\n5. 후보·ver4 평균·TF-IDF의 원시 확률로 Training calibration 내부의 혼합·임계값을 비교한다.\n6. 각 바깥 calibration fold의 평가 샘플은 안쪽의 혼합 선택과 임계값 학습에서 제외한다.\n7. 교차 평가에서 개선이 없으면 원래 ver4 구성으로 돌아가고, 선택 결과와 비교 점수를 함께 기록한다.\n8. 배포용 보정은 Training calibration 전체로 결정한 뒤 공식 Validation을 마지막에 한 번 평가한다.\n\n이미 본 공식 Validation 결과를 개선 방향의 근거로 참고하더라도, 그 정답으로 수치 설정을 최적화하지 않는다.\n배포 설정을 정한 자료에 다시 계산한 fitted F1은 교차 평가 F1보다 낙관적일 수 있어 별도로 표기한다.\n비교 후 선택 자체에도 불확실성이 있으므로 Training 교차 평가를 숨겨진 Test 성능이라고 설명하지 않는다.\n다섯 fold를 새로 학습하지 않는다. 기존 ensemble은 기준 모델이며 후보가 도움이 되는지 확인하는 대상이다.\n추론에 필요한 원래 fold·후보·sparse·tokenizer·manifest·실행 코드를 함께 보관해야 한다.\n\n## 6. 시간 예산과 결과 해석\n\nver4 저장 기록의 22 epoch 합은 630.28분, epoch 평균은 28.65분이었다.\n이 합에는 epoch별 학습과 stopping 평가가 포함되고, 저장·다운로드·최종 추론 시간은 빠져 있다.\nver5는 A100에서 최대 동결 1+미세조정 3 epoch와 보정·추론을 포함해 **대략 2–4시간을 목표**로 한다.\n이는 아직 실행하지 않은 추정이며 배치 처리, 데이터 위치, GPU 메모리, 저장 속도에 따라 달라진다.\n노트북은 실제 처리 속도에 따른 ETA를 출력한다. 추정치를 실측 완료 시간으로 보고하지 않는다.\n최종 보고에는 선택된 구성, 후보와 기준의 교차 평가, 공식 Validation 결과, 실제 시간을 구분한다.\n후보가 채택되지 않으면 “ver4 유지”라고 기록하며, 버전 이름만 바뀐 것을 성능 개선으로 표현하지 않는다.\n', 'ver4_runtime/README.md': '# Mission 3 ver4 실행 안내\n\n실행 파일은 [src/mission3_ver4.ipynb](../mission3_ver4.ipynb)입니다. 이 노트북 하나만 Colab에 올리면 필요한 모델·학습·추론 코드를 생성합니다. 이 폴더의 `.py` 파일은 같은 기능을 로컬이나 제출 환경에서 실행하기 위한 코드입니다.\n\n## 기존 결과와 개선 근거\n\n| 실행 버전 | 실제 Validation Macro F1 | 근거 |\n|---|---:|---|\n| `mission3_all_in_one_ver2` → results/mission3/ver1 | 0.600123 | `results/mission3/ver1/mission3_metrics.json` |\n| `mission3_ver3` → results/mission3/ver3 | 0.626015 | `results/mission3/ver3/mission3_ver3_metrics.json` |\n| ver4 | 아직 미측정 | 전체 GPU 학습 후 평가 필요 |\n\nver3의 Training 내부 F1은 3 epoch에서 0.644361로 최고였고, 15 epoch에는 0.601536으로 하락했습니다. 학습 손실은 계속 감소해 과적합 징후가 있습니다. ver4는 별도 stopping 집합의 Macro AP로 최적 epoch를 선택하고 조기 종료합니다. AP는 임계값 없이 예측 순위를 평가하며, 최종 대회 지표는 그대로 **9개 증상의 Macro F1**입니다.\n\nTraining 29,200개를 저장된 ver3 tokenizer로 직접 측정했을 때 384토큰을 넘는 대화는 3,292개(11.27%), 512토큰을 넘는 대화는 788개(2.70%)였습니다. 특수 토큰을 포함한 최대 길이는 873토큰입니다. ver4는 긴 대화의 뒷부분도 처리하도록 설계했습니다.\n\n**현재 작성 환경에는 GPU가 없어 ver4 전체 학습과 Validation 성능은 확인하지 못했습니다.** 작은 CPU 모델을 이용한 실행 검증은 코드 경로 확인용입니다. 기존 버전보다 성능이 높다는 판단은 전체 학습·추론이 끝난 뒤 실제 점수로 해야 합니다.\n\n## 모델과 데이터 분리\n\n- `klue/roberta-large`를 5개 fold에서 각각 학습하고 예측 확률을 평균냅니다.\n- 한 window는 최대 512토큰이며 인접 window는 64토큰이 겹칩니다. 마지막 window까지 유지하고, 증상별 attention으로 대화 전체를 모은 뒤 **통화당 한 번** BCE 손실을 계산합니다.\n- 문자 2–5그램과 단어 1–2그램 TF-IDF + 증상별 Logistic Regression을 함께 학습합니다.\n- 증상마다 신경망/TF-IDF의 혼합 비율과 판정 임계값을 Training 내부에서 결정합니다. 어떤 통화도 최소 1개 증상을 출력하도록 강제하지 않습니다.\n\n서울 Training을 약 90% development와 10% calibration으로 먼저 나눕니다. Development 내부 5-fold는 원래 Training 기준으로 각 fold마다 약 72% fit, 18% stopping입니다. 동일한 정규화 대화는 같은 그룹에 두어 분할을 넘지 않게 합니다.\n\nCalibration 10%는 신경망 학습, TF-IDF 학습, epoch 선택에 전혀 사용하지 않습니다. TF-IDF는 development 90%에만 fit합니다. 완성된 5개 신경망의 평균 확률과 TF-IDF 확률을 calibration에 적용한 뒤, 그 안에서 다시 5-fold 교차 보정하여 혼합 비율·임계값 선택의 일반화 점수를 계산합니다. 배포용 보정값은 calibration 전체로 결정합니다. **보정 후 기본 모델을 다시 학습하지 않고 같은 모델들을 그대로 추론에 사용합니다.** 공식 Validation은 이 과정에 들어가지 않습니다.\n\n주요 기본값은 `pipeline.py`의 `DEFAULT_CONFIG`에 있습니다.\n\n| 설정 | 기본값 |\n|---|---|\n| 모델 / fold | `klue/roberta-large` / 5 |\n| 최대 epoch / 조기 종료 patience | 6 / 2 |\n| 통화 batch / gradient accumulation | 2 / 16 |\n| encoder / head learning rate | `1.5e-5` / `1e-4` |\n| 문자 / 단어 최대 특징 수 | 120,000 / 40,000 |\n\n## Colab에서 실행\n\n1. [mission3_ver4.ipynb](../mission3_ver4.ipynb)를 Colab에 업로드합니다.\n2. GPU 런타임을 선택합니다. 기본 large 모델의 5-fold 학습에는 **A100 또는 L4를 권장**합니다. 사용 가능한 장치에 따라 시간과 메모리 사용량은 달라집니다.\n3. 제공 데이터의 압축을 풀고 `TL_서울_구급`, `VL_서울_구급` 폴더 이름을 유지합니다. Mission 3은 JSON만 읽습니다.\n4. 위에서부터 모든 셀을 실행하고 Google Drive를 연결합니다. 노트북은 `/content`를 먼저 탐색하고 Drive의 알려진 데이터 경로를 확인합니다. 자동 탐색에 실패하면 노트북의 데이터 경로 설정에 실제 폴더를 지정합니다.\n5. 학습·보정이 끝난 뒤 Validation 추론과 평가 셀을 실행합니다. 실행 로그가 남은 노트북도 저장합니다.\n\n기본 결과 폴더는 Colab에서 `/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4`, 로컬에서 프로젝트의 `results/mission3/ver4`입니다. 신경망 가중치 5개를 저장하므로 Drive 여유 공간도 확인합니다. 패키지와 최초 공개 pretrained 모델 다운로드에는 인터넷 연결이 필요하며, 학습 완료 후에는 저장된 구성으로 오프라인 추론할 수 있습니다.\n\n## 로컬 또는 제출 환경에서 실행\n\n다음 명령은 `DCC_AIdeas` 프로젝트 루트 기준입니다. 전체 학습 명령은 CUDA GPU를 요구합니다.\n\n```bash\npython -m pip install -r src/mission3_ver4/requirements.txt\n\npython src/mission3_ver4/train.py \\\n  --label_dir "DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급" \\\n  --output_dir results/mission3/ver4\n\npython src/mission3_ver4/inference.py \\\n  --audio_dir "DATA/대학부 데이터/Validation/1.원천데이터/VS_서울_구급" \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --ckpt_path results/mission3/ver4/mission3_ver4_model.ckpt \\\n  --output results/mission3/ver4/mission3.csv\n\npython src/mission3_ver4/evaluate.py \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --prediction results/mission3/ver4/mission3.csv \\\n  --output results/mission3/ver4/validation_metrics.json\n```\n\n`--audio_dir`는 주최 측 인터페이스를 맞추기 위해 받으며 음성 파일을 읽지 않습니다. 추론은 `--device cpu`도 지원하지만 large 모델 5개를 처리하므로 GPU보다 느릴 수 있습니다. 학습 설정은 `train.py --config 설정.json`으로 변경할 수 있습니다. 설정을 바꿀 때는 새 `--output_dir`를 사용합니다.\n\n## 결과 읽기와 파일 보관\n\n| 파일 | 의미 |\n|---|---|\n| `mission3_ver4_metrics.json` | Training 내부 보정 점수, 증상별 교차 보정 F1, 최종 보정값 |\n| `mission3_ver4_history.json` | 각 fold의 epoch별 학습·stopping 기록 |\n| `validation_metrics.json` | 위 평가 명령으로 만든 공식 Validation Macro F1와 증상별 점수 |\n| `mission3.csv` | `label file name,symptom` 형식의 예측 결과 |\n| `run.json`, `splits.json` | 실행 설정·데이터/코드 식별값과 분할 파일 목록 |\n| `submission/` | 실행 당시 `.py` 파일, requirements와 실행 안내 사본 |\n\n`calibration_crossfit_macro_f1`는 각 보정 평가 샘플의 정답을 보정값 선택에서 제외하여 계산한 **Training 내부 점수**입니다. `calibration_fit_macro_f1`는 같은 calibration 데이터에서 보정값을 정하고 다시 평가한 점수라 낙관적일 수 있습니다. 둘을 Validation 점수로 보고하면 안 됩니다. 공식 Validation 평가를 모델 선택이나 임계값 재조정에 반복 사용하지 않습니다.\n\n**`mission3_ver4_model.ckpt`는 구성 목록과 보정값을 담은 manifest입니다. 이 파일 하나만 옮기면 추론할 수 없습니다.** 결과 디렉터리 전체를 함께 보관하거나 최소한 다음 구성을 유지합니다.\n\n```text\nver4/\n├─ mission3_ver4_model.ckpt\n├─ fold_1.ckpt ... fold_5.ckpt\n├─ tokenizer/\n├─ sparse.joblib\n└─ submission/\n```\n\nCSV의 symptom 값은 기존 결과와 같은 문자열 목록 표현입니다. 예를 들어 `[]`, `[\'구토\', \'복통\']`처럼 저장하며 CSV 인용 처리는 코드가 담당합니다. 주최 측 sample submission에서 더 구체적인 형식을 제공하면 해당 규격도 확인합니다.\n\n## 중단 후 다시 실행\n\n같은 데이터·설정·코드와 같은 결과 폴더로 다시 실행하면 `fold_N.ckpt`와 `fold_N_done.json`이 모두 존재하는 **완료된 fold만 재사용**합니다. 실행 도중 끊긴 fold는 처음부터 다시 학습합니다. 마지막 미니배치나 optimizer 상태에서 이어가는 기능은 아닙니다.\n\n데이터·설정·코드의 식별값이 기존 실행과 다르면 혼합된 결과를 만들지 않도록 중단합니다. 이 경우 새 결과 폴더를 지정합니다. 메모리가 부족하면 통화 batch와 window batch를 줄이는 설정을 검토하고 새 폴더에서 시작합니다.\n\n## 대회 조건과 참고 자료\n\n판단 기준은 [docs의 출제 PDF](../../docs/0820_DCC%20대학부_예선_출제문제.pdf) 5쪽, 9–12쪽입니다. 서울 Training만 학습하고 양측의 전사 `text`만 입력으로 사용합니다. `speaker`, 시간, 주소, 성별, 정답 라벨은 입력 특징에서 제외합니다. 대상은 `고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란`이며, 나머지 라벨만 제거하고 통화는 유지합니다. 외부 추가 데이터와 상용 AI API를 사용하지 않습니다.\n\n최종 제출에는 추론 코드, 학습 로그 포함 노트북, 학습 완료 가중치, 모델 코드, requirements 및 결과 정리 PPT가 필요합니다. 이 구현의 학습을 완료하고 로그·가중치를 보관해야 하며 PPT는 별도로 준비합니다.\n\n공식 구현 참고: [KLUE RoBERTa large](https://huggingface.co/klue/roberta-large), [Transformers tokenizer](https://huggingface.co/docs/transformers/en/main_classes/tokenizer), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html). 모델 로딩·토큰 처리·혼합 정밀도 API 참고 자료이며 Mission 3의 성능 향상을 보장하는 근거는 아닙니다.\n', 'ver4_runtime/__init__.py': '"""Unmodified ver4 runtime retained for compatible inference and run verification."""\n', 'ver4_runtime/calibration.py': '"""Training-only grouped calibration for nine independent symptom labels."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .data import TARGET_SYMPTOMS\nelse:\n    from data import TARGET_SYMPTOMS\n\nWEIGHT_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nALL_NEGATIVE_THRESHOLD = float(np.nextafter(1.0, np.inf))\n\n\ndef _probabilities(values, name):\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS):\n        raise ValueError(f"{name} must have shape (samples, {len(TARGET_SYMPTOMS)})")\n    if not len(values) or not np.isfinite(values).all():\n        raise ValueError(f"{name} must be nonempty and finite")\n    if np.any((values < 0) | (values > 1)):\n        raise ValueError(f"{name} must contain probabilities in [0, 1]")\n    return values\n\n\ndef _inputs(y_true, neural_probs, sparse_probs):\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    truth = np.asarray(y_true)\n    if truth.shape != neural.shape or sparse.shape != neural.shape:\n        raise ValueError("truth, neural_probs and sparse_probs must have identical shapes")\n    if not np.isin(truth, (0, 1)).all():\n        raise ValueError("y_true must contain only binary labels")\n    return truth.astype(np.int8), neural, sparse\n\n\ndef _best_threshold(truth, probabilities):\n    """Exact F1 maximum for >= decisions; tied probabilities stay together."""\n    positives = int(truth.sum())\n    if positives == 0:\n        return ALL_NEGATIVE_THRESHOLD, 0.0\n    if positives == len(truth):\n        return 0.0, 1.0\n    order = np.argsort(-probabilities, kind="stable")\n    sorted_probs = probabilities[order]\n    # A threshold can include a complete tied block, never part of one.\n    boundaries = np.r_[np.flatnonzero(sorted_probs[:-1] != sorted_probs[1:]), len(truth) - 1]\n    true_positives = np.cumsum(truth[order])[boundaries]\n    scores = 2.0 * true_positives / (positives + boundaries + 1)\n    thresholds = np.zeros(len(boundaries), dtype=np.float64)\n    has_next = boundaries < len(truth) - 1\n    upper = sorted_probs[boundaries[has_next]]\n    lower = sorted_probs[boundaries[has_next] + 1]\n    midpoint = lower + (upper - lower) / 2\n    # Adjacent floating-point values may have no representable midpoint.\n    thresholds[has_next] = np.where(midpoint > lower, midpoint, upper)\n    best = np.flatnonzero(scores == scores.max())\n    # An explicit deterministic tie break preserves the exact maximum.\n    winner = min(best, key=lambda i: (abs(thresholds[i] - 0.5), -thresholds[i]))\n    return float(thresholds[winner]), float(scores[winner])\n\n\ndef fit_calibrator(y_true, neural_probs, sparse_probs):\n    """Select each label\'s blend and threshold using supplied training labels."""\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    weights, thresholds = [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        candidates = []\n        for weight in WEIGHT_GRID:\n            blended = weight * neural[:, column] + (1.0 - weight) * sparse[:, column]\n            threshold, score = _best_threshold(truth[:, column], blended)\n            candidates.append((score, -abs(weight - 0.5), weight, threshold))\n        _, _, weight, threshold = max(candidates)\n        weights.append(float(weight))\n        thresholds.append(float(threshold))\n    return {"neural_weights": weights, "thresholds": thresholds}\n\n\ndef apply_calibrator(neural_probs, sparse_probs, calibrator):\n    """Return blended probabilities; use >= stored thresholds for labels."""\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    if neural.shape != sparse.shape:\n        raise ValueError("neural_probs and sparse_probs must have identical shapes")\n    weights = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n    thresholds = np.asarray(calibrator["thresholds"], dtype=np.float64)\n    expected = (len(TARGET_SYMPTOMS),)\n    if weights.shape != expected or thresholds.shape != expected:\n        raise ValueError("calibrator must contain one weight and threshold per symptom")\n    if not np.isfinite(weights).all() or np.any((weights < 0) | (weights > 1)):\n        raise ValueError("calibrator weights must be finite and in [0, 1]")\n    if not np.isfinite(thresholds).all() or np.any((thresholds < 0) | (thresholds > ALL_NEGATIVE_THRESHOLD)):\n        raise ValueError("calibrator thresholds must be finite and in [0, nextafter(1, inf)]")\n    return weights * neural + (1.0 - weights) * sparse\n\n\ndef _grouped_splits(truth, groups, seed, n_splits):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) != len(truth):\n        raise ValueError("groups must contain one group ID per sample")\n    if not isinstance(n_splits, (int, np.integer)) or n_splits < 2:\n        raise ValueError("n_splits must be an integer >= 2")\n    unique_groups, inverse = np.unique(groups, return_inverse=True)\n    if len(unique_groups) < 2:\n        raise ValueError("cross-fit calibration requires at least two distinct groups")\n    n_splits = min(n_splits, len(unique_groups))\n    group_targets = np.zeros((len(unique_groups), truth.shape[1]), dtype=np.int8)\n    np.maximum.at(group_targets, inverse, truth)\n    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n    for train_groups, test_groups in splitter.split(np.zeros(len(unique_groups)), group_targets):\n        train = np.flatnonzero(np.isin(inverse, train_groups))\n        test = np.flatnonzero(np.isin(inverse, test_groups))\n        if not len(train) or not len(test):\n            raise ValueError("grouped calibration produced an empty fold")\n        yield train, test\n\n\ndef calibrate_crossfit(y_true, neural_probs, sparse_probs, groups, seed=42, n_splits=5):\n    """Cross-fit within Training calibration data, then fit deployment settings.\n\n    Base-model predictions must themselves come from samples excluded from\n    base-model training and model selection. Never pass official Validation.\n    """\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    crossfit_predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        calibrator = fit_calibrator(truth[train], neural[train], sparse[train])\n        probabilities = apply_calibrator(neural[test], sparse[test], calibrator)\n        crossfit_predictions[test] = probabilities >= np.asarray(calibrator["thresholds"])\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("each calibration sample must be scored exactly once")\n    final_calibrator = fit_calibrator(truth, neural, sparse)\n    fitted = apply_calibrator(neural, sparse, final_calibrator) >= np.asarray(final_calibrator["thresholds"])\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, crossfit_predictions, average=None, zero_division=0\n    )\n    report = {\n        "score_scope": "Training-only calibration; cross-fit estimates calibration generalization, not official Validation.",\n        "calibration_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "calibration_crossfit_macro_f1": float(f1.mean()),\n        "per_symptom_crossfit": {\n            symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n            for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n    }\n    return final_calibrator, report\n', 'ver4_runtime/data.py': '"""Mission 3: 모델 입력은 양측 발화의 text만, symptom은 학습/평가 정답만."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport unicodedata\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """입력/정답과 무관하게 동일하게 전처리하며 발화 순서를 유지한다."""\n    utterances = annotation.get("utterances")\n    if not isinstance(utterances, list):\n        utterances = annotation.get("utterences", [])\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if isinstance(utterance, dict) and isinstance(utterance.get("text"), str):\n            text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n            if text:\n                texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(x, str) for x in raw):\n        raise ValueError("symptom must be a string or a list of strings")\n    found = {x.strip() for x in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    # 파일명, 날짜, speaker 등은 모델 특징에도 중복 판단에도 사용하지 않는다.\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir: Path, include_targets: bool = False, workers: int = 8):\n    paths = sorted(Path(label_dir).rglob("*.json"))\n    if not paths:\n        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")\n    if len({p.name for p in paths}) != len(paths):\n        raise ValueError("동일한 JSON 파일명이 중복됩니다. 제출 CSV의 식별자가 모호합니다.")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"JSON object가 아닙니다: {path}")\n        # 추론 경로에서는 target_vector를 호출하거나 symptom에 접근하지 않는다.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef grouped_targets(texts, targets):\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse = np.unique(groups, return_inverse=True)\n    aggregate = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    np.maximum.at(aggregate, inverse, targets)\n    return groups, inverse, aggregate\n\n\ndef make_splits(texts, targets, calibration_size: float, n_folds: int, seed: int):\n    """전체 Training의 보정 집합은 모든 모델 fit/epoch 선택에서 분리한다."""\n    if not 0 < calibration_size < 0.5 or n_folds < 2:\n        raise ValueError("calibration_size must be in (0, .5); n_folds >= 2")\n    groups, inverse, aggregate = grouped_targets(texts, targets)\n    group_ids = np.arange(len(aggregate))\n    outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=calibration_size, random_state=seed)\n    dev_groups, cal_groups = next(outer.split(group_ids, aggregate))\n    development = np.flatnonzero(np.isin(inverse, dev_groups))\n    calibration = np.flatnonzero(np.isin(inverse, cal_groups))\n    splitter = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + 1)\n    folds = []\n    for train_g, stop_g in splitter.split(dev_groups, aggregate[dev_groups]):\n        train_indices = np.flatnonzero(np.isin(inverse, dev_groups[train_g]))\n        stop_indices = np.flatnonzero(np.isin(inverse, dev_groups[stop_g]))\n        folds.append((train_indices, stop_indices))\n    for name, indices in [("calibration", calibration), ("development", development),\n                          *[(f"fold{i}/{part}", ids) for i, pair in enumerate(folds)\n                            for part, ids in zip(("fit", "stop"), pair)]]:\n        counts = targets[indices].sum(axis=0)\n        if len(indices) == 0 or np.any(counts == 0) or np.any(counts == len(indices)):\n            raise ValueError(f"{name}: 각 증상의 양성/음성이 필요합니다. 데이터/분할 수를 확인하세요.")\n    return development, calibration, folds, groups\n\n\ndef write_json(path: Path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_predictions(paths, probabilities, thresholds, output: Path):\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(paths), len(TARGET_SYMPTOMS)) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid prediction shape or non-finite probabilities")\n    predicted = probabilities >= np.asarray(thresholds)\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predicted):\n            # 기존 제출 CSV와 같은 Python string-list 표현. 빈 list도 허용한다.\n            writer.writerow([Path(path).name, str([s for s, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    return predicted.astype(np.int8)\n\n\ndef score_predictions(targets, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i])}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n', 'ver4_runtime/evaluate.py': '"""Score frozen CSV predictions after inference, without changing the model."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .data import write_json\n    from .pipeline import evaluate_csv\nelse:\n    from data import write_json\n    from pipeline import evaluate_csv\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--prediction", required=True, type=Path)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    report = evaluate_csv(args.label_dir, args.prediction)\n    if args.output:\n        write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/inference.py': '"""Official one-command inference. Mission 3 accepts but does not read audio_dir."""\nimport argparse\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import predict_pipeline\nelse:\n    from pipeline import predict_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", required=True, help="Organizer interface; unused text-only task")\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--ckpt_path", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--device", choices=["cpu", "cuda"])\n    parser.add_argument("--batch_size", type=int)\n    args = parser.parse_args()\n    predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size)\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/neural.py': '"""Document-level multi-label transformer; every dialogue window is retained."""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nimport random\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\n\nNUM_LABELS = 9\n\n\ndef tokenize_dialogues(texts, tokenizer, max_length=512, stride=64):\n    """Return documents of token windows, including the last window of long calls.\n\n    Only ordinary tokenizer special tokens are used. The stride is an overlap in\n    tokens, not an upper bound on the number of windows or dialogue length.\n    """\n    usable_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    if usable_length < 1 or not 0 <= stride < usable_length:\n        raise ValueError("max_length must fit special tokens; 0 <= stride < usable length")\n    texts = list(texts)\n    documents = [[] for _ in texts]\n    for start in range(0, len(texts), 128):\n        encoded = tokenizer(\n            texts[start:start + 128], truncation=True, max_length=max_length,\n            stride=stride, return_overflowing_tokens=True, padding=False,\n            return_attention_mask=True,\n        )\n        if "overflow_to_sample_mapping" not in encoded:\n            raise ValueError("Use a fast tokenizer (AutoTokenizer(..., use_fast=True))")\n        for window, sample in enumerate(encoded["overflow_to_sample_mapping"]):\n            documents[start + int(sample)].append({\n                key: encoded[key][window]\n                for key in tokenizer.model_input_names if key in encoded\n            })\n    if any(not windows for windows in documents):\n        raise ValueError("Tokenizer returned a document without any windows")\n    return documents\n\n\nclass DialogueDataset(Dataset):\n    def __init__(self, documents, targets=None):\n        self.documents, self.targets = documents, targets\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["labels"] = self.targets[index]\n        return item\n\n\nclass DialogueCollator:\n    """Flatten windows for the encoder while keeping exactly one target per call."""\n\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, examples):\n        windows, doc_indices = [], []\n        for index, example in enumerate(examples):\n            if not example["windows"]:\n                raise ValueError("A document must contain at least one token window")\n            windows.extend(example["windows"])\n            doc_indices.extend([index] * len(example["windows"]))\n        batch = dict(self.tokenizer.pad(windows, padding=True, return_tensors="pt"))\n        batch["doc_indices"] = torch.tensor(doc_indices, dtype=torch.long)\n        if "labels" in examples[0]:\n            batch["labels"] = torch.from_numpy(np.asarray(\n                [example["labels"] for example in examples], dtype=np.float32,\n            ))\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Label-specific attention pools complete dialogues before computing BCE."""\n\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=8):\n        super().__init__()\n        self.encoder = encoder\n        self.window_batch_size = int(window_batch_size)\n        if self.window_batch_size < 1:\n            raise ValueError("window_batch_size must be positive")\n        hidden = encoder.config.hidden_size\n        self.projection = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh())\n        self.dropout = nn.Dropout(dropout)\n        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        nn.init.normal_(self.attention_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, doc_indices, **inputs):\n        representations = []\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            window_inputs = {key: value[start:start + self.window_batch_size]\n                             for key, value in inputs.items()}\n            hidden = self.encoder(**window_inputs).last_hidden_state\n            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))\n        windows = self.dropout(torch.cat(representations))\n        logits = []\n        # Each label learns which part of a potentially long call provides evidence.\n        for document_index in range(int(doc_indices.max().item()) + 1):\n            selected = windows[doc_indices == document_index]\n            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])\n            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)\n            pooled = attention.T @ selected\n            logits.append((self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias)\n        return torch.stack(logits)\n\n\ndef _device(config):\n    device = torch.device(config.get("device", "cuda" if torch.cuda.is_available() else "cpu"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime.")\n    return device\n\n\ndef _amp(device):\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.amp.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\ndef _loader(documents, targets, tokenizer, batch_size, shuffle=False, seed=0):\n    if int(batch_size) < 1:\n        raise ValueError("batch_size must be positive")\n    generator = torch.Generator().manual_seed(seed)\n    return DataLoader(\n        DialogueDataset(documents, targets), batch_size=int(batch_size),\n        shuffle=shuffle, collate_fn=DialogueCollator(tokenizer),\n        num_workers=0, generator=generator,\n    )\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _optimizer(model, config):\n    depth = int(getattr(model.encoder.config, "num_hidden_layers", 0))\n    decay = float(config.get("layerwise_lr_decay", 0.9))\n    if not 0 < decay <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        if name.startswith("encoder."):\n            match = re.search(r"\\.layer\\.(\\d+)\\.", name)\n            level = int(match.group(1)) + 1 if match else (0 if ".embeddings." in name else depth + 1)\n            lr = float(config.get("learning_rate", 1.5e-5)) * decay ** (depth + 1 - level)\n        else:\n            lr = float(config.get("head_learning_rate", 1e-4))\n        wd = 0.0 if parameter.ndim < 2 or name.endswith("bias") else float(config.get("weight_decay", 0.01))\n        groups.setdefault((lr, wd), []).append(parameter)\n    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd}\n                             for (lr, wd), params in groups.items()])\n\n\ndef _atomic_save(checkpoint, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(checkpoint, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _targets(targets, documents, name):\n    targets = np.asarray(targets, dtype=np.float32)\n    if targets.shape != (len(documents), NUM_LABELS) or not len(documents):\n        raise ValueError(f"{name} requires nonempty documents and targets of shape (N, {NUM_LABELS})")\n    if not np.isin(targets, [0, 1]).all():\n        raise ValueError(f"{name} targets must contain only 0 and 1")\n    return targets\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets,\n                tokenizer, config: dict, checkpoint_path: Path, seed: int):\n    """Train on documents; select best epoch only by stopping-set macro AP.\n\n    This function does not tune thresholds and never uses calibration targets.\n    GPU enforcement for production belongs to the orchestration entry point.\n    """\n    train_targets = _targets(train_targets, train_documents, "train")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = _device(config)\n    max_epochs, patience = int(config.get("max_epochs", 8)), int(config.get("patience", 2))\n    accumulation = int(config.get("gradient_accumulation_steps", 4))\n    if min(max_epochs, patience, accumulation) < 1:\n        raise ValueError("max_epochs, patience and gradient_accumulation_steps must be positive")\n    model_name = config.get("model_name", "klue/roberta-large")\n    encoder = AutoModel.from_pretrained(model_name)\n    if config.get("gradient_checkpointing", True):\n        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    architecture = {"num_labels": NUM_LABELS, "dropout": float(config.get("dropout", 0.2)),\n                    "window_batch_size": int(config.get("window_batch_size", 8))}\n    model = DialogueClassifier(encoder, **architecture).to(device)\n    train_loader = _loader(train_documents, train_targets, tokenizer,\n                           config.get("batch_size", 2), shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, config.get("inference_batch_size", 4))\n    positives = train_targets.sum(axis=0)\n    if (positives == 0).any():\n        raise ValueError(f"Training split has no positive examples for labels {np.flatnonzero(positives == 0).tolist()}")\n    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)\n                         ** float(config.get("pos_weight_power", 0.5)), 1, 5)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    optimizer = _optimizer(model, config)\n    total_steps = math.ceil(len(train_loader) / accumulation) * max_epochs\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(total_steps * float(config.get("warmup_ratio", 0.1))),\n        num_training_steps=total_steps,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    valid_labels = stop_targets.sum(axis=0) > 0\n    if not valid_labels.any():\n        raise ValueError("Stopping split has no positive labels; macro AP cannot select an epoch")\n    if not valid_labels.all():\n        print(f"Warning: stop AP excludes labels without positives: {np.flatnonzero(~valid_labels).tolist()}", flush=True)\n    best_ap, best_epoch, stale, history = -1.0, 0, 0, []\n    for epoch in range(1, max_epochs + 1):\n        epoch_started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, document_count = 0.0, 0\n        for step, batch in enumerate(train_loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            # Match a single full document batch, including a shorter last batch.\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * train_loader.batch_size,\n                                  len(train_documents) - group_start * train_loader.batch_size)\n            with _amp(device):\n                loss = criterion(model(**batch).float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite training loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += loss.detach().item() * len(labels)\n            document_count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            log_every = int(config.get("log_every", 100))\n            if log_every > 0 and (step + 1) % log_every == 0:\n                elapsed = (time.monotonic() - epoch_started) / 60\n                print(f"seed={seed} epoch={epoch} batch={step + 1}/{len(train_loader)} "\n                      f"loss={loss_sum / document_count:.5f} minutes={elapsed:.1f}", flush=True)\n        probabilities = _predict(model, stop_loader, device)\n        if not np.isfinite(probabilities).all():\n            raise FloatingPointError("Stopping predictions contain non-finite values")\n        score = float(average_precision_score(stop_targets[:, valid_labels],\n                                              probabilities[:, valid_labels], average="macro"))\n        row = {"epoch": epoch, "train_loss": loss_sum / document_count,\n               "stop_macro_ap": score, "stop_labels_with_positives": int(valid_labels.sum()),\n               "minutes": (time.monotonic() - epoch_started) / 60}\n        history.append(row)\n        print(f"seed={seed} epoch={epoch}/{max_epochs} loss={row[\'train_loss\']:.5f} "\n              f"stop_AP={score:.5f} minutes={row[\'minutes\']:.1f}", flush=True)\n        if score > best_ap + 1e-6:\n            best_ap, best_epoch, stale = score, epoch, 0\n            _atomic_save({\n                "format_version": 1, "encoder_config": encoder.config.to_dict(),\n                "architecture": architecture, "model_name": str(model_name),\n                "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                "epoch": epoch, "stop_ap": score, "seed": int(seed),\n            }, checkpoint_path)\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history}\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Rebuild from the saved config and weights; no pretrained network fetch."""\n    device = _device(config)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if checkpoint.get("format_version") != 1:\n        raise ValueError(f"Unsupported neural checkpoint format: {checkpoint_path}")\n    encoder_data = dict(checkpoint["encoder_config"])\n    model_type = encoder_data.pop("model_type")\n    encoder_config = AutoConfig.for_model(model_type, **encoder_data)\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(AutoModel.from_config(encoder_config), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    del checkpoint\n    model.to(device)\n    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 4))\n    predictions = _predict(model, loader, device)\n    if not np.isfinite(predictions).all():\n        raise FloatingPointError("Inference predictions contain non-finite values")\n    return predictions\n', 'ver4_runtime/pipeline.py': '"""Training-only model selection, ensemble calibration, and portable inference."""\nfrom __future__ import annotations\n\nimport ast\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import FeatureUnion\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from .calibration import apply_calibrator, calibrate_crossfit\n    from .data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from .neural import predict_checkpoint, tokenize_dialogues, train_model\nelse:\n    from calibration import apply_calibrator, calibrate_crossfit\n    from data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from neural import predict_checkpoint, tokenize_dialogues, train_model\n\n\nDEFAULT_CONFIG = {\n    "model_name": "klue/roberta-large",\n    "seed": 42,\n    "n_folds": 5,\n    "calibration_size": 0.10,\n    "calibration_folds": 5,\n    "max_length": 512,\n    "stride": 64,\n    "max_epochs": 6,\n    "patience": 2,\n    "batch_size": 2,\n    "inference_batch_size": 8,\n    "window_batch_size": 2,\n    "gradient_accumulation_steps": 16,\n    "learning_rate": 1.5e-5,\n    "head_learning_rate": 1e-4,\n    "layerwise_lr_decay": 0.95,\n    "weight_decay": 0.01,\n    "warmup_ratio": 0.10,\n    "dropout": 0.20,\n    "pos_weight_power": 0.5,\n    "gradient_checkpointing": True,\n    "device": "cuda",\n    "require_cuda": True,\n    "char_features": 120_000,\n    "word_features": 40_000,\n    "min_df": 2,\n    "logistic_c": 4.0,\n}\nRUNTIME_FILES = ("data.py", "calibration.py", "neural.py", "pipeline.py", "train.py", "inference.py", "evaluate.py")\n\n\ndef fit_sparse(texts, targets, config):\n    """문자 표현과 단어 순서를 보완하는, Training에만 fit한 TF-IDF 모델."""\n    features = FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                max_features=config["char_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\\b\\w+\\b",\n                                max_features=config["word_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n    matrix = features.fit_transform(texts)\n    classifiers = []\n    for column, name in enumerate(TARGET_SYMPTOMS):\n        truth = targets[:, column]\n        if len(np.unique(truth)) == 1:\n            classifier = float(truth[0])\n        else:\n            classifier = LogisticRegression(C=config["logistic_c"], class_weight="balanced",\n                                            solver="liblinear", max_iter=1500, random_state=config["seed"])\n            classifier.fit(matrix, truth)\n        classifiers.append(classifier)\n        print(f"TF-IDF fit: {name}", flush=True)\n    return {"features": features, "classifiers": classifiers}\n\n\ndef predict_sparse(model, texts):\n    if not texts:\n        return np.empty((0, len(TARGET_SYMPTOMS)), dtype=np.float32)\n    # 큰 Test 폴더도 전체 sparse matrix를 RAM에 동시에 올리지 않는다.\n    batches = []\n    for start in range(0, len(texts), 1024):\n        matrix = model["features"].transform(texts[start:start + 1024])\n        columns = [np.full(matrix.shape[0], clf) if isinstance(clf, float)\n                   else clf.predict_proba(matrix)[:, 1] for clf in model["classifiers"]]\n        batches.append(np.column_stack(columns))\n    return np.concatenate(batches).astype(np.float32)\n\n\ndef _release_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _fingerprint(paths, texts, targets, config):\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())\n    for path, text, target in zip(paths, texts, targets):\n        digest.update(json.dumps([path.name, text, target.tolist()], ensure_ascii=False).encode("utf-8"))\n    # 변경된 코드로 기존 fold가 조용히 재사용되는 것을 방지한다.\n    for name in RUNTIME_FILES:\n        path = Path(__file__).parent / name\n        digest.update(path.name.encode())\n        digest.update(path.read_bytes())\n    return digest.hexdigest()\n\n\ndef train_pipeline(train_label_dir, output_dir, config=None):\n    """Validation 인자를 받지 않는다. 보정 후에도 모델을 재학습하지 않는다."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    train_label_dir, output_dir = Path(train_label_dir).resolve(), Path(output_dir).resolve()\n    if train_label_dir.name != "TL_서울_구급" or "Validation" in train_label_dir.parts:\n        raise ValueError("학습 입력은 서울 Training의 TL_서울_구급 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab GPU 런타임에서 실행하세요.")\n    if config["device"] == "cuda":\n        torch.set_float32_matmul_precision("high")\n        print("GPU:", torch.cuda.get_device_name(0), flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    dev, calibration, folds, groups = make_splits(\n        texts, targets, config["calibration_size"], config["n_folds"], config["seed"],\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    signature = _fingerprint(paths, texts, targets, config)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8"))["signature"] != signature:\n        raise ValueError("출력 폴더에 다른 데이터/설정/코드의 실험이 있습니다. 새 output_dir를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "training_samples": len(paths)})\n    write_json(output_dir / "splits.json", {\n        "calibration": [paths[i].name for i in calibration],\n        "folds": [{"fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop]}\n                  for fit, stop in folds],\n        "seed": config["seed"], "grouping": "SHA256 of whitespace-normalized transcript only",\n    })\n    print(f"Training={len(paths):,}, development={len(dev):,}, calibration={len(calibration):,}", flush=True)\n    tokenizer_dir = output_dir / "tokenizer"\n    if (tokenizer_dir / "tokenizer_config.json").exists():\n        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)\n        tokenizer.save_pretrained(tokenizer_dir)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    calibration_documents = [documents[i] for i in calibration]\n    counts = np.asarray([len(document) for document in documents])\n    print(f"대화 window 수: 평균={counts.mean():.2f}, 최대={counts.max()}, 긴 대화={int((counts > 1).sum()):,}", flush=True)\n    neural_probs = np.zeros((len(calibration), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    fold_names, histories = [], []\n    for number, (fit, stop) in enumerate(folds, 1):\n        name = f"fold_{number}.ckpt"\n        checkpoint_path, done_path = output_dir / name, output_dir / f"fold_{number}_done.json"\n        print(f"\\nFold {number}/{len(folds)}: fit={len(fit):,}, stop={len(stop):,}", flush=True)\n        if done_path.exists() and checkpoint_path.exists():\n            result = json.loads(done_path.read_text(encoding="utf-8"))\n            print("완료된 fold 재사용 (동일 run signature 확인됨)", flush=True)\n        else:\n            result = train_model([documents[i] for i in fit], targets[fit],\n                                 [documents[i] for i in stop], targets[stop], tokenizer,\n                                 config, checkpoint_path, seed=config["seed"] + number * 1009)\n            write_json(done_path, result)\n        _release_memory()\n        neural_probs += predict_checkpoint(checkpoint_path, calibration_documents, tokenizer, config)\n        _release_memory()\n        fold_names.append(name)\n        histories.append(result)\n    neural_probs /= len(folds)\n    sparse_path = output_dir / "sparse.joblib"\n    if sparse_path.exists():\n        sparse = joblib.load(sparse_path)\n    else:\n        sparse = fit_sparse([texts[i] for i in dev], targets[dev], config)\n        temporary = sparse_path.with_name("sparse.joblib.tmp")\n        joblib.dump(sparse, temporary, compress=3)\n        temporary.replace(sparse_path)\n    sparse_probs = predict_sparse(sparse, [texts[i] for i in calibration])\n    del sparse\n    calibrator, report = calibrate_crossfit(targets[calibration], neural_probs, sparse_probs,\n                                          groups[calibration], seed=config["seed"] + 2,\n                                          n_splits=config["calibration_folds"])\n    # OOF 확률을 final refit에 전용하지 않고, 실제 배포 ensemble의 확률을 보정한다.\n    np.savez_compressed(output_dir / "calibration_predictions.npz", targets=targets[calibration],\n                        neural=neural_probs, sparse=sparse_probs,\n                        filenames=np.asarray([paths[i].name for i in calibration]))\n    versions = {name: importlib.metadata.version(name) for name in\n                ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n    manifest = {"format": "mission3_ver4", "version": 1, "target_symptoms": list(TARGET_SYMPTOMS),\n                "config": config, "folds": fold_names, "tokenizer": "tokenizer", "sparse": "sparse.joblib",\n                "calibrator": calibrator, "signature": signature, "versions": versions}\n    checkpoint_path = output_dir / "mission3_ver4_model.ckpt"\n    _torch_save(checkpoint_path, manifest)\n    report.update({"training_samples": len(paths), "development_samples": len(dev),\n                   "model": f"{config[\'model_name\']} {len(folds)}-fold ensemble + char/word TF-IDF LR",\n                   "calibrator": calibrator, "fold_best_epochs": [x["best_epoch"] for x in histories],\n                   "versions": versions, "validation_used_for_training_or_calibration": False})\n    write_json(output_dir / "mission3_ver4_metrics.json", report)\n    write_json(output_dir / "mission3_ver4_history.json", histories)\n    # 노트북 하나만 Colab에 올려도 제출 코드와 실행 당시 설정을 남긴다.\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in (*RUNTIME_FILES, "requirements.txt", "README.md"):\n        path = Path(__file__).parent / name\n        if path.exists():\n            shutil.copy2(path, submission / path.name)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    print("Checkpoint:", checkpoint_path, flush=True)\n    return checkpoint_path, report\n\n\ndef load_manifest(checkpoint_path):\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver4" or manifest.get("version") != 1:\n        raise ValueError("지원하지 않는 checkpoint입니다. ver4의 manifest .ckpt를 지정하세요.")\n    if manifest["target_symptoms"] != list(TARGET_SYMPTOMS) or not manifest["folds"]:\n        raise ValueError("Checkpoint class order/folds mismatch")\n    root = Path(checkpoint_path).resolve().parent\n    for name in [*manifest["folds"], manifest["sparse"], manifest["tokenizer"]]:\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.exists():\n            raise FileNotFoundError(f"Checkpoint 구성 파일을 함께 보관해야 합니다: {name}")\n    return manifest\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):\n    """저장된 encoder config/tokenizer만 사용하므로 학습 후 오프라인 추론 가능."""\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = batch_size\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(root / manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    neural_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for name in manifest["folds"]:\n        print("Predict:", name, flush=True)\n        neural_probs += predict_checkpoint(root / name, documents, tokenizer, config)\n        _release_memory()\n    neural_probs /= len(manifest["folds"])\n    sparse = joblib.load(root / manifest["sparse"])\n    sparse_probs = predict_sparse(sparse, texts)\n    probabilities = apply_calibrator(neural_probs, sparse_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\ndef evaluate_csv(label_dir, prediction_path):\n    """이미 저장된 예측을 파일명으로 정답에 연결한다. 모델/임계값 변경 없음."""\n    with Path(prediction_path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != ["label file name", "symptom"]:\n            raise ValueError("CSV header must be: label file name,symptom")\n        rows = list(reader)\n    by_name = {}\n    for row in rows:\n        if row["label file name"] in by_name:\n            raise ValueError("Duplicate prediction filename")\n        labels = ast.literal_eval(row["symptom"])\n        if not isinstance(labels, list) or any(x not in TARGET_SYMPTOMS for x in labels) or len(set(labels)) != len(labels):\n            raise ValueError("CSV symptom must be a list of unique target strings")\n        by_name[row["label file name"]] = [int(x in labels) for x in TARGET_SYMPTOMS]\n    paths, _, targets = load_dataset(Path(label_dir), include_targets=True)\n    if set(by_name) != {path.name for path in paths}:\n        raise ValueError("CSV와 JSON 파일명 집합이 일치하지 않습니다.")\n    return score_predictions(targets, np.asarray([by_name[path.name] for path in paths], dtype=np.int8))\n', 'ver4_runtime/requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy>=1.26,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'ver4_runtime/train.py': '"""Run all Training-only fitting and calibration; never accepts Validation."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--output_dir", required=True, type=Path)\n    parser.add_argument("--config", type=Path, help="Optional JSON overrides of DEFAULT_CONFIG")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding="utf-8")) if args.config else None\n    train_pipeline(args.label_dir, args.output_dir, config)\n\n\nif __name__ == "__main__":\n    main()\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver5_runtime_"))
for name, source in RUNTIME_SOURCES.items():
    path = runtime_dir / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
config_path = runtime_dir / "notebook_config.json"
config_path.write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")

def run_python(arguments):
    # Stream child output into the notebook; stopping the cell also stops the worker.
    command = [sys.executable, "-u", *map(str, arguments)]
    environment = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", env=environment)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except BaseException:
        process.kill()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print("실행 코드:", runtime_dir)


## 5. 추가 학습과 Training 내부 비교

이 단계에는 **Training만 전달**합니다. 저장된 split·데이터·모델 파일의 일치를 먼저 검사합니다.
첫 무결성 검사에는 약 6.4GiB의 기존 가중치를 읽는 시간이 필요합니다.
`phase=warmup`은 encoder 고정 학습, `phase=finetune`은 전체 미세조정입니다.
`epoch=0 baseline`은 학습 전 stop 평가입니다. 후보가 이를 넘지 못하면 epoch 0이 최적으로 남을 수 있습니다.

혼합 비율 비교는 이미 계산한 확률로 수행하는 CPU 작업입니다. 기존 5개 모델을 다시 학습하지 않습니다.
학습 후 결과 폴더를 독립적으로 제출할 수 있도록 `base/`에 기존 가중치를 복사합니다.

중간에 끊겼다면 같은 설정으로 모두 실행하세요. 마지막 **완료 epoch**에서 복원하며 중단된 epoch는 재실행합니다.
설정·데이터·코드가 바뀌면 다른 실험으로 취급하므로 새 `OUTPUT_DIR`가 필요합니다.


In [ ]:
run_python([
    runtime_dir / "train.py", "--train_label_dir", TRAIN_LABEL_DIR,
    "--base_dir", BASE_DIR, "--output_dir", OUTPUT_DIR,
    "--config_json", config_path, "--auto_batch",
])
CHECKPOINT_PATH = OUTPUT_DIR / "mission3_ver5_model.ckpt"


In [ ]:
# 6. Training 내부 선택 결과 — 이 점수는 공식 Validation 점수가 아닙니다.
training_report = json.loads((OUTPUT_DIR / "mission3_ver5_metrics.json").read_text(encoding="utf-8"))
print("새 모델의 선택 epoch:", training_report["candidate_training"]["best_epoch"])
print("기존 ver4 교차평가:", training_report["baseline_crossfit_macro_f1"])
print("ver5 후보 교차평가:", training_report["candidate_crossfit_macro_f1"])
print("최종 선택:", training_report["selected_name"])
if training_report["selected_name"] == "ver4_baseline":
    print("추가 모델이 내부 비교에서 개선되지 않아 기존 ver4 보정을 사용합니다.")
print("증상 | 기존 신경망 alpha | 새 후보 beta | 임계값")
labels = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
selected = training_report["calibrator"]
for name, alpha, beta, threshold in zip(labels, selected["neural_weights"], selected["candidate_weights"], selected["thresholds"]):
    print(f"{name} | {alpha:.2f} | {beta:.2f} | {threshold:.9f}")
print("계산에는 저장된 보정값의 전체 정밀도를 사용합니다.")


## 7. 고정된 최종 모델의 예측과 평가

여기부터 처음으로 공식 Validation/Test 입력을 사용합니다. JSON의 전사 text만 모델에 전달하고,
CSV를 저장한 뒤에만 symptom 정답을 읽어 평가합니다. 이 결과로 모델·보정값을 다시 선택하지 않습니다.
필요하다고 선택된 모델만 추론하며, 기존 5개와 새 후보가 모두 선택되면 최대 6개 모델의 추론이 필요합니다.

GPU·batch에 따른 수치 차이가 있을 수 있습니다. 기존 전체 Validation Macro F1은 ver4 **0.656649**이며,
새 결과와의 실제 차이는 아래 최종 평가에서 확인하세요.


In [ ]:
PREDICTION_PATH = None
if RUN_VALIDATION and VALID_LABEL_DIR is not None:
    # Preserve previous evaluation outputs when rerunning or changing the input folder.
    stamp = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
    prediction_dir = OUTPUT_DIR / "predictions" / (stamp + "_" + uuid.uuid4().hex[:6])
    PREDICTION_PATH = prediction_dir / "mission3.csv"
    run_python([
        runtime_dir / "inference.py", "--audio_dir", "", "--label_dir", VALID_LABEL_DIR,
        "--ckpt_path", CHECKPOINT_PATH, "--output", PREDICTION_PATH,
        "--batch_size", INFERENCE_BATCH_SIZE, "--evaluate",
    ])
    metrics_path = prediction_dir / "validation_metrics.json"
    if metrics_path.is_file():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(f"현재 입력 {metrics['samples']:,}개 Macro F1: {metrics['macro_f1']:.6f}")
else:
    print("학습·모델 저장 완료. RUN_VALIDATION=False 또는 예측 폴더가 없어 최종 추론을 생략했습니다.")
print("모델 묶음:", OUTPUT_DIR)
print("manifest:", CHECKPOINT_PATH)
print("CSV:", PREDICTION_PATH)
print("제출용 CLI:", OUTPUT_DIR / "submission/inference.py")


## 결과 보관

- `mission3_ver5_model.ckpt`: 선택된 구성과 보정값.
- `candidate_best.ckpt`: 새 모델의 최적 epoch 가중치.
- `candidate_best_last.ckpt`: 완료 epoch 재개 상태. 추론에는 필요하지 않습니다.
- `base/`: 재사용한 기존 모델 5개, tokenizer, TF-IDF 모델.
- `mission3_ver5_metrics.json`: 후보 학습 이력, 교차평가 비교와 선택 이유.
- `predictions/실행시각/mission3.csv`: 제출 형식의 예측. 정답이 있으면 같은 폴더에 `validation_metrics.json`.
- `submission/`: 모델 코드, 추론 CLI, requirements, 논문 검토 문서.

**학습 로그가 남은 이 노트북도 저장하세요.** 결과 폴더 전체를 보관하면 다른 환경에서도 아래 명령으로 실행할 수 있습니다.

```bash
python submission/inference.py --audio_dir "WAV 폴더" --label_dir "JSON 폴더" \
  --ckpt_path mission3_ver5_model.ckpt --output mission3.csv
```

추론만 다시 할 때는 학습이 필요하지 않습니다. 대회가 요구한 결과 정리 PPT는 별도 작성 대상입니다.
